In [ ]:
# ============================================================
# 0) GOOGLE DRIVE BAĞLANTISI
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
Dataset 1 - UofM NMC Pouch Cell Voltage and Expansion Cyclic Aging Dataset
Multi-HI + MAW-GME + 7 Structured Scenario Pipeline
=====================================================

Google Colab-ready script.

Expected Drive folder structure:

/content/drive/MyDrive/Batarya/Dataset1_NMC_MAW_GME/
    raw/
        data.zip                  # optional, script can unzip it
        data/                     # optional, if already extracted
            01/OCV_wExpansion.csv
            03/OCV_wExpansion.csv
            10/OCV_wExpansion.csv
            11/OCV_wExpansion.csv
            12/OCV_wExpansion.csv

The script primarily uses OCV_wExpansion.csv because it contains periodic C/20
charge-discharge characterization tests, which are cleaner for ICA feature extraction.

Outputs:
    features/multi_hi_features.csv
    results/single_hi_polynomial_ekf_*.csv
    results/multi_hi_tabular_*.csv
    results/maw_gme_*.csv
    results/structured_window_ablation_*.csv
    results/advanced_soh_experiment_report.xlsx

Author note:
    This script is adapted for Dataset 1 after Dataset 2 and Dataset 3 MAW-GME pipelines.
"""

In [ ]:
# ============================================================
# 1) INSTALL / IMPORT DEPENDENCIES
# ============================================================
import os
import re
import sys
import math
import glob
import zipfile
import warnings
import random
import subprocess
import importlib.util
from pathlib import Path

warnings.filterwarnings("ignore")


def pip_install_if_missing(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, imp in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("matplotlib", "matplotlib"),
    ("scikit-learn", "sklearn"),
    ("openpyxl", "openpyxl"),
    ("tqdm", "tqdm"),
]:
    pip_install_if_missing(pkg, imp)

# Optional tree boosting packages. If installation fails, script continues.
OPTIONAL_PACKAGES = [
    ("xgboost", "xgboost"),
    ("lightgbm", "lightgbm"),
    ("catboost", "catboost"),
]
for pkg, imp in OPTIONAL_PACKAGES:
    try:
        pip_install_if_missing(pkg, imp)
    except Exception as e:
        print(f"Optional package {pkg} could not be installed: {e}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.signal import savgol_filter
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None

try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor = None

In [ ]:
# ============================================================
# 2) PATHS AND SETTINGS
# ============================================================

import os
from pathlib import Path

# Dataset_1 altında gerçek cell klasörlerini otomatik bulacağız
SEARCH_ROOT = "/content/drive/MyDrive/Batarya/Dataset_1"

# Paper-selected Dataset 1 cells
TARGET_CELLS = ["01", "03", "10", "11", "12"]

USE_OCV_FILE = True
SOURCE_FILE_NAME = "OCV_wExpansion.csv" if USE_OCV_FILE else "cycling_wExpansion.csv"

print("SEARCH_ROOT:", SEARCH_ROOT)
print("SEARCH_ROOT exists:", os.path.exists(SEARCH_ROOT))
print("Looking for source file:", SOURCE_FILE_NAME)

# ------------------------------------------------------------
# Gerçek cell klasörlerini bul
# ------------------------------------------------------------
found_cell_dirs = {}

for root, dirs, files in os.walk(SEARCH_ROOT):
    base = os.path.basename(root)

    if base in TARGET_CELLS:
        file_names_lower = [f.lower() for f in files]

        if SOURCE_FILE_NAME.lower() in file_names_lower:
            found_cell_dirs[base] = root

print("\nFound cell folders:")
for cell in TARGET_CELLS:
    print(cell, "->", found_cell_dirs.get(cell, "NOT FOUND"))

# ------------------------------------------------------------
# Tüm hedef cell klasörleri bulundu mu?
# ------------------------------------------------------------
if not all(cell in found_cell_dirs for cell in TARGET_CELLS):
    print("\nSome target cell folders were not found.")
    print("Searching all OCV_wExpansion.csv files under SEARCH_ROOT...")

    ocv_files = []
    for root, dirs, files in os.walk(SEARCH_ROOT):
        for f in files:
            if f.lower() == SOURCE_FILE_NAME.lower():
                ocv_files.append(os.path.join(root, f))

    print("\nFound OCV files:")
    for p in ocv_files[:50]:
        print(" -", p)

    raise RuntimeError(
        "Could not find all target cell folders with the requested source file. "
        "Check the printed OCV file paths above."
    )

# ------------------------------------------------------------
# Ortak ana DATA_DIR klasörünü bul
# Örnek:
# DATA_DIR/
#     01/OCV_wExpansion.csv
#     03/OCV_wExpansion.csv
#     ...
# ------------------------------------------------------------
parent_dirs = [os.path.dirname(found_cell_dirs[cell]) for cell in TARGET_CELLS]
unique_parent_dirs = sorted(set(parent_dirs))

print("\nUnique parent directories:")
for p in unique_parent_dirs:
    print(" -", p)

if len(unique_parent_dirs) != 1:
    raise RuntimeError(
        "Target cell folders are not under the same parent directory. "
        "Please check the folder structure."
    )

DATA_DIR = unique_parent_dirs[0]

# ------------------------------------------------------------
# Sonuçları ayrı bir klasöre yazalım
# Böylece veri klasörü ile output klasörü karışmaz
# ------------------------------------------------------------
PROJECT_DIR = os.path.join(SEARCH_ROOT, "Dataset1_NMC_MAW_GME_results")

FEATURE_DIR = os.path.join(PROJECT_DIR, "features")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")
FIG_DIR = os.path.join(PROJECT_DIR, "figures")

for d in [PROJECT_DIR, FEATURE_DIR, RESULT_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# Dataset 1 windows
WINDOWS = {
    "full_32_42": (3.20, 4.20),
    "medium_35_40": (3.50, 4.00),
    "narrow_36_385": (3.60, 3.85),
    "very_narrow_365_38": (3.65, 3.80),
}

# Main settings
USE_CYCLE_NUMBER = True
USE_GLOBAL_TEMPERATURE = True
USE_EXPANSION_FEATURES = False
RUN_DEEP_MODELS = False  # Set True to run LSTM/GRU/TCN/Transformer; keep same in all datasets for fair deep comparison.
DEEP_SEQ_LEN = 5
DEEP_EPOCHS = 80
DEEP_PATIENCE = 10
DEEP_BATCH_SIZE = 32
DEEP_LR = 1e-3
RUN_SHAP = True
RANDOM_STATE = 42

FEATURE_OUT_CSV = os.path.join(FEATURE_DIR, "multi_hi_features.csv")

print("\nDataset 1 data directory:", DATA_DIR)
print("Project/output directory:", PROJECT_DIR)
print("Feature directory:", FEATURE_DIR)
print("Result directory:", RESULT_DIR)
print("Figure directory:", FIG_DIR)
print("Target cells:", TARGET_CELLS)
print("Source file:", SOURCE_FILE_NAME)
print("Feature output CSV:", FEATURE_OUT_CSV)
print("Partial windows:", WINDOWS)

# ------------------------------------------------------------
# Final file check
# ------------------------------------------------------------
print("\nFinal check:")

for cell in TARGET_CELLS:
    cell_dir = os.path.join(DATA_DIR, cell)
    source_path = os.path.join(cell_dir, SOURCE_FILE_NAME)

    print("\nCell:", cell)
    print("Cell folder exists:", os.path.exists(cell_dir))
    print("Source file exists:", os.path.exists(source_path))
    print("Source path:", source_path)

In [ ]:
# ============================================================
# 4) DATA LOADING HELPERS
# ============================================================
COLUMN_NAMES_9 = [
    "time_s",
    "current_mA",
    "voltage_V",
    "expansion_um",
    "temperature_C",
    "Q_Ah",
    "capacity_Ah_raw",
    "cycle_number",
    "total_Ah_throughput_Ah",
]

COLUMN_NAMES_8 = COLUMN_NAMES_9[:-1]


def find_cell_file(cell_id, file_name=SOURCE_FILE_NAME):
    """
    Find the requested Dataset 1 CSV for a given cell id.

    The PATHS cell already auto-detects DATA_DIR as the parent folder that
    directly contains the cell folders, for example:
        DATA_DIR/01/OCV_wExpansion.csv
        DATA_DIR/03/OCV_wExpansion.csv

    Therefore, this function must not use RAW_DIR. RAW_DIR is intentionally
    not defined in the aligned Dataset 1 pipeline because outputs are written
    to a separate PROJECT_DIR.
    """
    cell_id = str(cell_id).zfill(2)

    # 1) Direct expected path
    direct_path = os.path.join(DATA_DIR, cell_id, file_name)
    if os.path.isfile(direct_path):
        return direct_path

    # 2) Case-insensitive search inside the cell folder
    cell_dir = os.path.join(DATA_DIR, cell_id)
    if os.path.isdir(cell_dir):
        for root, dirs, files in os.walk(cell_dir):
            for f in files:
                if f.lower() == file_name.lower():
                    return os.path.join(root, f)

    # 3) More flexible search under DATA_DIR, but only inside the relevant cell folder
    candidates = []
    for root, dirs, files in os.walk(DATA_DIR):
        parts = root.replace("\\", "/").split("/")
        if cell_id not in parts:
            continue
        for f in files:
            if f.lower() == file_name.lower():
                candidates.append(os.path.join(root, f))

    if candidates:
        return sorted(candidates)[0]

    raise FileNotFoundError(
        f"Could not find {file_name} for Cell{cell_id}. "
        f"Expected path: {direct_path}"
    )


def read_uofm_csv(path):
    """Read UofM CSV that has one header row and numeric columns."""
    # MATLAB starter code uses csvread(strFull, 1, 0), so skip first row.
    df = pd.read_csv(path, skiprows=1, header=None)
    if df.shape[1] >= 9:
        df = df.iloc[:, :9]
        df.columns = COLUMN_NAMES_9
    elif df.shape[1] == 8:
        df.columns = COLUMN_NAMES_8
        df["total_Ah_throughput_Ah"] = np.nan
    else:
        raise ValueError(f"Unexpected number of columns in {path}: {df.shape[1]}")

    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["time_s", "current_mA", "voltage_V", "cycle_number"]).copy()
    return df


def split_by_cycle_number(df):
    groups = []
    for cyc, g in df.groupby("cycle_number", sort=True):
        g = g.sort_values("time_s").reset_index(drop=True)
        if len(g) >= 20:
            groups.append((cyc, g))
    return groups


In [ ]:
# ============================================================
# 5) SEGMENT EXTRACTION AND FEATURE ENGINEERING
# ============================================================
def contiguous_true_segments(mask):
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0:
        return []
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return []
    splits = np.where(np.diff(idx) > 1)[0] + 1
    parts = np.split(idx, splits)
    return [(int(p[0]), int(p[-1])) for p in parts if len(p) > 1]


def choose_charge_segment(g):
    """
    Detect the CC charge segment automatically.
    Candidate segment should have nonzero current and increasing voltage.
    This avoids assuming a fixed current sign.
    """
    v = g["voltage_V"].values.astype(float)
    i = g["current_mA"].values.astype(float)

    finite = np.isfinite(v) & np.isfinite(i)
    if finite.sum() < 20:
        return None

    i_abs = np.abs(i)
    nonzero_thr = max(5.0, 0.05 * np.nanpercentile(i_abs[finite], 95))

    candidates = []
    for sign in [-1, 1]:
        mask = finite & (np.sign(i) == sign) & (i_abs > nonzero_thr)
        for s, e in contiguous_true_segments(mask):
            if e - s + 1 < 20:
                continue
            vv = v[s:e+1]
            ii = i[s:e+1]
            dv = float(np.nanmax(vv) - np.nanmin(vv))
            net = float(vv[-1] - vv[0])
            # Charge segment should be voltage-increasing.
            if dv < 0.4 or net < 0.3:
                continue
            # Exclude CV tail by trimming once voltage reaches near upper limit.
            sub = g.iloc[s:e+1].copy().reset_index(drop=True)
            sub = sub[sub["voltage_V"] <= 4.205].copy()
            if len(sub) < 20:
                continue
            # Prefer wide voltage span and stable current.
            current_cv = np.nanstd(np.abs(ii)) / (np.nanmean(np.abs(ii)) + 1e-9)
            score = dv - 0.1 * current_cv
            candidates.append((score, sub))

    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1].reset_index(drop=True)


def compute_discharge_capacity_ah(g, charge_segment=None):
    """Compute C/20 discharge capacity in Ah from the largest decreasing-voltage nonzero-current segment."""
    v = g["voltage_V"].values.astype(float)
    i = g["current_mA"].values.astype(float)
    t = g["time_s"].values.astype(float)
    finite = np.isfinite(v) & np.isfinite(i) & np.isfinite(t)
    if finite.sum() < 20:
        return np.nan
    i_abs = np.abs(i)
    nonzero_thr = max(5.0, 0.05 * np.nanpercentile(i_abs[finite], 95))

    candidates = []
    for sign in [-1, 1]:
        mask = finite & (np.sign(i) == sign) & (i_abs > nonzero_thr)
        for s, e in contiguous_true_segments(mask):
            if e - s + 1 < 20:
                continue
            vv = v[s:e+1]
            net = float(vv[-1] - vv[0])
            span = float(np.nanmax(vv) - np.nanmin(vv))
            if span < 0.4 or net > -0.3:
                continue
            tt = t[s:e+1]
            ii = i_abs[s:e+1] / 1000.0  # A
            dt_h = np.diff(tt, prepend=tt[0]) / 3600.0
            dt_h = np.where((dt_h > 0) & np.isfinite(dt_h), dt_h, 0.0)
            cap = float(np.nansum(ii * dt_h))
            candidates.append((cap, s, e))

    # Fallback from raw capacity column.
    cap_col = g["capacity_Ah_raw"].values.astype(float)
    cap_col_valid = cap_col[np.isfinite(cap_col)]
    fallback = np.nan
    if len(cap_col_valid) > 0:
        fallback = float(np.nanmax(np.abs(cap_col_valid)))

    if not candidates:
        return fallback

    best_cap = max(candidates, key=lambda x: x[0])[0]
    if np.isfinite(fallback) and fallback > 0.1:
        # Use a conservative max of integration and recorded capacity.
        return float(max(best_cap, fallback))
    return float(best_cap)


def charge_q_from_current_or_Q(ch):
    t = ch["time_s"].values.astype(float)
    i_A = np.abs(ch["current_mA"].values.astype(float)) / 1000.0
    q_raw = ch["Q_Ah"].values.astype(float)
    v = ch["voltage_V"].values.astype(float)

    # Integrated charge in Ah.
    dt_h = np.diff(t, prepend=t[0]) / 3600.0
    dt_h = np.where((dt_h > 0) & np.isfinite(dt_h), dt_h, 0.0)
    q_int = np.cumsum(i_A * dt_h)

    # Raw Q can be running counter. Use if it has a reasonable monotonic range.
    if np.isfinite(q_raw).sum() >= 10:
        q_rel = q_raw - q_raw[0]
        if np.nanstd(q_rel) > 1e-6:
            corr = np.corrcoef(np.nan_to_num(q_rel), np.nan_to_num(v))[0, 1]
            if np.isfinite(corr) and corr < 0:
                q_rel = -q_rel
            if np.nanmax(q_rel) - np.nanmin(q_rel) > 0.05:
                q_rel = q_rel - np.nanmin(q_rel)
                return q_rel

    return q_int


def smooth_vector(x, window=15, poly=3):
    x = np.asarray(x, dtype=float)
    if len(x) < 7:
        return x
    # Window must be odd and <= len(x).
    w = min(window, len(x) if len(x) % 2 == 1 else len(x) - 1)
    if w < 7:
        return x
    try:
        return savgol_filter(x, window_length=w, polyorder=min(poly, w-2), mode="interp")
    except Exception:
        return x


def compute_window_features(v, q_ah, t, temp, expansion, window_name, vmin, vmax):
    """Compute ICA/DVA/charge-time/q-delta features for a voltage window."""
    out = {}
    prefix = f"_{window_name}"

    v = np.asarray(v, dtype=float)
    q_ah = np.asarray(q_ah, dtype=float)
    t = np.asarray(t, dtype=float)
    temp = np.asarray(temp, dtype=float) if temp is not None else np.full_like(v, np.nan)
    expansion = np.asarray(expansion, dtype=float) if expansion is not None else np.full_like(v, np.nan)

    finite = np.isfinite(v) & np.isfinite(q_ah)
    if finite.sum() < 20:
        for key in ["pmax", "pmax_voltage", "ic_area", "ic_mean", "ic_std", "peak_width",
                    "charge_time_window", "q_delta_window", "dva_mean", "dva_std", "dva_abs_mean",
                    "temp_mean_window", "temp_max_window"]:
            out[f"{key}{prefix}"] = np.nan
        return out

    v = v[finite]
    q_ah = q_ah[finite]
    t = t[finite] if len(t) == len(finite) else np.full(len(v), np.nan)
    temp = temp[finite] if len(temp) == len(finite) else np.full(len(v), np.nan)
    expansion = expansion[finite] if len(expansion) == len(finite) else np.full(len(v), np.nan)

    # Sort by voltage and remove duplicate voltage values.
    order = np.argsort(v)
    v = v[order]
    q_ah = q_ah[order]
    t = t[order]
    temp = temp[order]
    expansion = expansion[order]

    # Aggregate duplicated voltage by keeping first occurrence after small rounding.
    v_round = np.round(v, 5)
    _, unique_idx = np.unique(v_round, return_index=True)
    unique_idx = np.sort(unique_idx)
    v = v[unique_idx]
    q_ah = q_ah[unique_idx]
    t = t[unique_idx]
    temp = temp[unique_idx]
    expansion = expansion[unique_idx]

    mask_win = (v >= vmin) & (v <= vmax)
    if mask_win.sum() < 8:
        for key in ["pmax", "pmax_voltage", "ic_area", "ic_mean", "ic_std", "peak_width",
                    "charge_time_window", "q_delta_window", "dva_mean", "dva_std", "dva_abs_mean",
                    "temp_mean_window", "temp_max_window"]:
            out[f"{key}{prefix}"] = np.nan
        return out

    v_s = smooth_vector(v, window=15, poly=3)
    q_s_mAh = smooth_vector(q_ah * 1000.0, window=15, poly=3)

    # Central finite difference.
    dv = np.gradient(v_s)
    dq = np.gradient(q_s_mAh)
    ic = dq / (dv + 1e-9)
    ic = np.where(np.isfinite(ic), ic, np.nan)

    # If sign is mostly negative, flip. ICA for charge should be positive.
    if np.nanmedian(ic[mask_win]) < 0:
        ic = -ic

    win_ic = ic[mask_win]
    win_v = v_s[mask_win]
    win_q = q_s_mAh[mask_win]
    win_t = t[mask_win]
    win_temp = temp[mask_win]
    win_exp = expansion[mask_win]

    valid_ic = np.isfinite(win_ic) & np.isfinite(win_v)
    if valid_ic.sum() < 5:
        for key in ["pmax", "pmax_voltage", "ic_area", "ic_mean", "ic_std", "peak_width",
                    "charge_time_window", "q_delta_window", "dva_mean", "dva_std", "dva_abs_mean",
                    "temp_mean_window", "temp_max_window"]:
            out[f"{key}{prefix}"] = np.nan
        return out

    win_ic = win_ic[valid_ic]
    win_v = win_v[valid_ic]
    win_q = win_q[valid_ic]
    win_t = win_t[valid_ic]
    win_temp = win_temp[valid_ic]
    win_exp = win_exp[valid_ic]

    # Robustly avoid extreme derivative spikes.
    if len(win_ic) > 10:
        lo, hi = np.nanpercentile(win_ic, [1, 99])
        win_ic_clip = np.clip(win_ic, lo, hi)
    else:
        win_ic_clip = win_ic

    pmax_idx = int(np.nanargmax(win_ic_clip))
    pmax = float(win_ic_clip[pmax_idx])
    pmax_voltage = float(win_v[pmax_idx])
    ic_area = float(np.trapz(win_ic_clip, win_v)) if len(win_v) > 1 else np.nan
    ic_mean = float(np.nanmean(win_ic_clip))
    ic_std = float(np.nanstd(win_ic_clip))

    half = 0.5 * pmax
    above = win_ic_clip >= half
    if above.sum() >= 2:
        peak_width = float(np.nanmax(win_v[above]) - np.nanmin(win_v[above]))
    else:
        peak_width = np.nan

    if np.isfinite(win_t).sum() >= 2:
        charge_time_window = float(np.nanmax(win_t) - np.nanmin(win_t))
    else:
        charge_time_window = np.nan

    q_delta_window = float(abs(np.nanmax(win_q) - np.nanmin(win_q))) if len(win_q) > 1 else np.nan
    dva = 1.0 / (win_ic_clip + 1e-9)
    dva = np.where(np.isfinite(dva), dva, np.nan)

    out[f"pmax{prefix}"] = pmax
    out[f"pmax_voltage{prefix}"] = pmax_voltage
    out[f"ic_area{prefix}"] = ic_area
    out[f"ic_mean{prefix}"] = ic_mean
    out[f"ic_std{prefix}"] = ic_std
    out[f"peak_width{prefix}"] = peak_width
    out[f"charge_time_window{prefix}"] = charge_time_window
    out[f"q_delta_window{prefix}"] = q_delta_window
    out[f"dva_mean{prefix}"] = float(np.nanmean(dva))
    out[f"dva_std{prefix}"] = float(np.nanstd(dva))
    out[f"dva_abs_mean{prefix}"] = float(np.nanmean(np.abs(dva)))
    out[f"temp_mean_window{prefix}"] = float(np.nanmean(win_temp)) if np.isfinite(win_temp).any() else np.nan
    out[f"temp_max_window{prefix}"] = float(np.nanmax(win_temp)) if np.isfinite(win_temp).any() else np.nan

    if USE_EXPANSION_FEATURES:
        out[f"expansion_mean_window{prefix}"] = float(np.nanmean(win_exp)) if np.isfinite(win_exp).any() else np.nan
        out[f"expansion_delta_window{prefix}"] = float(np.nanmax(win_exp) - np.nanmin(win_exp)) if np.isfinite(win_exp).any() else np.nan

    return out


def extract_features_for_cell(cell_id):
    path = find_cell_file(cell_id, SOURCE_FILE_NAME)
    print(f"Reading Cell{cell_id}: {path}")
    df_raw = read_uofm_csv(path)
    groups = split_by_cycle_number(df_raw)

    rows = []
    for idx, (cyc, g) in enumerate(tqdm(groups, desc=f"Feature extraction Cell{cell_id}")):
        charge = choose_charge_segment(g)
        if charge is None or len(charge) < 25:
            continue

        cap_ah = compute_discharge_capacity_ah(g, charge)
        if not np.isfinite(cap_ah) or cap_ah <= 0.05:
            continue

        v = charge["voltage_V"].values.astype(float)
        q_ah = charge_q_from_current_or_Q(charge)
        t = charge["time_s"].values.astype(float)
        temp = charge["temperature_C"].values.astype(float) if "temperature_C" in charge.columns else np.full(len(v), np.nan)
        expansion = charge["expansion_um"].values.astype(float) if "expansion_um" in charge.columns else np.full(len(v), np.nan)

        row = {
            "cell": f"Cell{cell_id}",
            "cycle_name": f"cyc_{int(cyc):05d}" if np.isfinite(cyc) else f"idx_{idx:05d}",
            "rpt_index": idx,
            "cycle_number": float(cyc) if np.isfinite(cyc) else float(idx),
            "capacity_Ah": cap_ah,
            "capacity_mAh": cap_ah * 1000.0,
            "temp_mean": float(np.nanmean(temp)) if np.isfinite(temp).any() else np.nan,
            "temp_max": float(np.nanmax(temp)) if np.isfinite(temp).any() else np.nan,
            "temp_min": float(np.nanmin(temp)) if np.isfinite(temp).any() else np.nan,
            "temp_std": float(np.nanstd(temp)) if np.isfinite(temp).any() else np.nan,
            "n_charge_points": len(charge),
            "v_min_charge": float(np.nanmin(v)),
            "v_max_charge": float(np.nanmax(v)),
            "source_file": SOURCE_FILE_NAME,
        }

        if USE_EXPANSION_FEATURES:
            row["expansion_mean"] = float(np.nanmean(expansion)) if np.isfinite(expansion).any() else np.nan
            row["expansion_delta"] = float(np.nanmax(expansion) - np.nanmin(expansion)) if np.isfinite(expansion).any() else np.nan

        for wname, (vmin, vmax) in WINDOWS.items():
            row.update(compute_window_features(v, q_ah, t, temp, expansion, wname, vmin, vmax))

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# 6) BUILD OR LOAD FEATURE TABLE
# ============================================================
FORCE_REBUILD_FEATURES = True

if os.path.exists(FEATURE_OUT_CSV) and not FORCE_REBUILD_FEATURES:
    df = pd.read_csv(FEATURE_OUT_CSV)
    print("Loaded existing feature table:", FEATURE_OUT_CSV)
else:
    all_dfs = []
    for cid in TARGET_CELLS:
        try:
            cell_df = extract_features_for_cell(cid)
            if not cell_df.empty:
                all_dfs.append(cell_df)
        except Exception as e:
            print(f"FAILED Cell{cid}: {e}")

    if len(all_dfs) == 0:
        raise RuntimeError("No features were extracted. Check Dataset 1 folder structure and CSV files.")

    df = pd.concat(all_dfs, ignore_index=True)

    # SoH per cell from first valid capacity.
    df = df.sort_values(["cell", "cycle_number", "rpt_index"]).reset_index(drop=True)
    soh_values = []
    for cell, g in df.groupby("cell", sort=False):
        first_cap = g["capacity_Ah"].dropna().iloc[0]
        soh_values.extend((g["capacity_Ah"] / first_cap).values)
    df["soh"] = soh_values

    # Normalize pmax within each cell and window.
    for w in WINDOWS:
        p_col = f"pmax_{w}"
        norm_col = f"pmax_norm_{w}"
        if p_col in df.columns:
            df[norm_col] = np.nan
            for cell, g in df.groupby("cell"):
                valid = g[p_col].dropna()
                if len(valid) > 0 and abs(valid.iloc[0]) > 1e-12:
                    df.loc[g.index, norm_col] = g[p_col] / valid.iloc[0]

    # Outlier helper columns, not model inputs.
    df["soh_rolling_median"] = (
        df.groupby("cell")["soh"]
        .transform(lambda s: s.rolling(window=5, center=True, min_periods=1).median())
    )
    df["soh_delta"] = df.groupby("cell")["soh"].diff()
    df["soh_outlier"] = (df["soh"] - df["soh_rolling_median"]).abs() > 0.05

    # Basic sanity filter.
    df = df[(df["soh"] <= 1.05) & (df["soh"] >= 0.50)].copy()
    df.to_csv(FEATURE_OUT_CSV, index=False)
    print("Saved feature table:", FEATURE_OUT_CSV)

print("Feature table shape:", df.shape)
display(df.head())

# Save correlation with SoH.
corr_rows = []
for c in df.columns:
    if c in ["soh"] or df[c].dtype == object:
        continue
    valid = df[[c, "soh"]].dropna()
    if len(valid) >= 10:
        corr = valid[c].corr(valid["soh"])
        corr_rows.append({"feature": c, "corr_with_soh": corr, "abs_corr": abs(corr), "valid_count": len(valid)})

corr_df = pd.DataFrame(corr_rows).sort_values("abs_corr", ascending=False)
corr_df.to_csv(os.path.join(FEATURE_DIR, "multi_hi_correlation_with_soh.csv"), index=False)
print("Top correlated features with SoH:")
display(corr_df.head(20))

cell_summary = df.groupby("cell").agg(
    n_rpt=("soh", "size"),
    first_cycle=("cycle_number", "min"),
    last_cycle=("cycle_number", "max"),
    first_capacity_Ah=("capacity_Ah", "first"),
    last_capacity_Ah=("capacity_Ah", "last"),
    first_soh=("soh", "first"),
    last_soh=("soh", "last"),
    outlier_count=("soh_outlier", "sum"),
).reset_index()
print("Cell-level summary:")
display(cell_summary)
cell_summary.to_csv(os.path.join(FEATURE_DIR, "cell_level_summary.csv"), index=False)

In [ ]:
# ============================================================
# 7) MODELING HELPERS
# ============================================================
LEAKAGE_COLUMNS_EXACT = {
    "capacity_mAh", "capacity_Ah", "capacity_Ah_raw", "soh", "soh_delta",
    "soh_rolling_median", "soh_outlier", "cycle_name", "source_file"
}

EXCLUDE_OUTLIER_OPTIONS = [False, True]

# Common candidate pool used across Dataset 1, Dataset 2 and Dataset 3.
# Keeping this identical prevents unfair cross-dataset comparisons.
MODEL_CANDIDATES = [
    "Ridge",
    "ExtraTrees",
    "RandomForest",
    "HistGB",
    "XGBoost",
    "LightGBM",
    "CatBoost",
]


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)) if len(np.unique(y_true)) > 1 else np.nan,
    }


def make_preprocess_model(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])


def make_tabular_models():
    models = {
        "Ridge": make_preprocess_model(Ridge(alpha=1.0, random_state=RANDOM_STATE)),
        "ElasticNet": make_preprocess_model(ElasticNet(alpha=0.0005, l1_ratio=0.2, max_iter=20000, random_state=RANDOM_STATE)),
        "SVR": make_preprocess_model(SVR(kernel="rbf", C=20.0, gamma="scale", epsilon=0.002)),
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, min_samples_leaf=1, n_jobs=-1)),
        ]),
        "ExtraTrees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, min_samples_leaf=1, n_jobs=-1)),
        ]),
        "HistGB": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(max_iter=250, learning_rate=0.04, random_state=RANDOM_STATE)),
        ]),
    }
    if XGBRegressor is not None:
        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=350, learning_rate=0.03, max_depth=3,
                subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE,
                objective="reg:squarederror"
            )),
        ])
    if LGBMRegressor is not None:
        models["LightGBM"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=RANDOM_STATE, verbose=-1)),
        ])
    if CatBoostRegressor is not None:
        models["CatBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostRegressor(iterations=300, learning_rate=0.03, depth=4, random_seed=RANDOM_STATE, verbose=False)),
        ])
    return models


def unique_preserve_order(cols):
    return list(dict.fromkeys(cols))


def remove_unusable_feature_cols(dataframe, cols, scenario_name=""):
    cols = unique_preserve_order([c for c in cols if c in dataframe.columns])
    all_nan_cols = [c for c in cols if dataframe[c].isna().all()]
    if all_nan_cols:
        print(f"[{scenario_name}] removed all-NaN feature columns:", all_nan_cols)
    cols = [c for c in cols if c not in all_nan_cols]
    bad = [c for c in cols if c in LEAKAGE_COLUMNS_EXACT or c.startswith("soh") or c in ["capacity_mAh", "capacity_Ah"]]
    if bad:
        raise ValueError(f"Leakage columns found in {scenario_name}: {bad}")
    if len(cols) == 0:
        raise ValueError(f"No usable features for {scenario_name}")
    return cols


def get_window_feature_cols(dataframe, window_name, use_cycle_number=True, use_global_temperature=True):
    suffix = f"_{window_name}"
    cols = [c for c in dataframe.columns if c.endswith(suffix)]
    # Include pmax_norm explicitly because it is created after initial suffix features.
    pnorm = f"pmax_norm_{window_name}"
    if pnorm in dataframe.columns:
        cols.append(pnorm)
    if use_cycle_number and "cycle_number" in dataframe.columns:
        cols.append("cycle_number")
    if use_global_temperature:
        for c in ["temp_mean", "temp_max", "temp_min", "temp_std"]:
            if c in dataframe.columns:
                cols.append(c)
    if USE_EXPANSION_FEATURES:
        for c in ["expansion_mean", "expansion_delta"]:
            if c in dataframe.columns:
                cols.append(c)
    return remove_unusable_feature_cols(dataframe, cols, scenario_name=f"window::{window_name}")


def get_all_window_feature_cols(dataframe):
    cols = []
    for w in WINDOWS:
        cols.extend(get_window_feature_cols(dataframe, w, USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE))
    return remove_unusable_feature_cols(dataframe, unique_preserve_order(cols), scenario_name="all_windows")


def first_window_by_prefix(prefix):
    candidates = [w for w in WINDOWS if w.startswith(prefix)]
    if not candidates:
        raise ValueError(f"No window starts with {prefix}. WINDOWS={list(WINDOWS)}")
    return candidates[0]


def apply_structured_feature_filter(cols, scenario):
    cols = unique_preserve_order(cols)
    if scenario == "pmax_features_missing":
        return [c for c in cols if "pmax" not in c.lower()]
    if scenario == "ica_shape_features_missing":
        tokens = ["ic_std", "peak_width", "dva"]
        return [c for c in cols if not any(tok in c.lower() for tok in tokens)]
    return cols

In [ ]:
# ============================================================
# 8) SINGLE-HI POLYNOMIAL BASELINE
# ============================================================
def evaluate_single_hi_polynomial(dataframe):
    rows = []
    cells = sorted(dataframe["cell"].unique())
    for exclude_outliers in EXCLUDE_OUTLIER_OPTIONS:
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()
        for w in WINDOWS:
            feat = f"pmax_norm_{w}"
            if feat not in data_eval.columns:
                continue
            for test_cell in cells:
                train_df = data_eval[data_eval["cell"] != test_cell].dropna(subset=[feat, "soh"]).copy()
                test_df = data_eval[data_eval["cell"] == test_cell].dropna(subset=[feat, "soh"]).copy()
                if len(train_df) < 10 or len(test_df) < 5:
                    continue
                X_train = train_df[[feat]].values
                y_train = train_df["soh"].values
                X_test = test_df[[feat]].values
                y_test = test_df["soh"].values
                model = Pipeline([
                    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
                    ("ridge", Ridge(alpha=1e-4)),
                ])
                model.fit(X_train, y_train)
                pred = np.clip(model.predict(X_test), 0.5, 1.05)
                rows.append({
                    "experiment": "Single-HI-Polynomial",
                    "window": w,
                    "test_cell": test_cell,
                    "model": "Single-HI-Polynomial",
                    "exclude_outliers": exclude_outliers,
                    "n_train": len(train_df),
                    "n_test": len(test_df),
                    "feature": feat,
                    **regression_metrics(y_test, pred)
                })
    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(RESULT_DIR, "single_hi_polynomial_ekf_leave_one_cell_results.csv"), index=False)
    if not res.empty:
        summary = res.groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]].agg(["mean", "std"]).reset_index()
    else:
        summary = pd.DataFrame()
    summary.to_csv(os.path.join(RESULT_DIR, "single_hi_polynomial_ekf_summary_mean_std.csv"), index=False)
    return res, summary

# ============================================================
# 9) APPROX PAPER-LIKE GPR BASELINE
# ============================================================
def evaluate_approx_paper_gpr(dataframe):
    rows = []
    cells = sorted(dataframe["cell"].unique())
    for exclude_outliers in EXCLUDE_OUTLIER_OPTIONS:
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()
        for w in WINDOWS:
            feat = f"pmax_norm_{w}"
            if feat not in data_eval.columns:
                continue
            for test_cell in cells:
                train_df = data_eval[data_eval["cell"] != test_cell].dropna(subset=[feat, "cycle_number", "soh"]).copy()
                test_df = data_eval[data_eval["cell"] == test_cell].dropna(subset=[feat, "cycle_number", "soh"]).copy()
                if len(train_df) < 10 or len(test_df) < 5:
                    continue
                # Approximation: use cycle_number + normalized Pmax as GPR inputs.
                X_train = train_df[["cycle_number", feat]].values.astype(float)
                y_train = train_df["soh"].values.astype(float)
                X_test = test_df[["cycle_number", feat]].values.astype(float)
                y_test = test_df["soh"].values.astype(float)
                kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(2), length_scale_bounds=(1e-2, 1e3)) + WhiteKernel(noise_level=1e-4)
                model = Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                    ("gpr", GaussianProcessRegressor(kernel=kernel, alpha=1e-5, normalize_y=True, random_state=RANDOM_STATE, n_restarts_optimizer=2)),
                ])
                try:
                    model.fit(X_train, y_train)
                    pred = np.clip(model.predict(X_test), 0.5, 1.05)
                    rows.append({
                        "experiment": "Approx-Paper-GPR",
                        "window": w,
                        "test_cell": test_cell,
                        "model": "Approx-Paper-GPR",
                        "exclude_outliers": exclude_outliers,
                        "n_train": len(train_df),
                        "n_test": len(test_df),
                        **regression_metrics(y_test, pred)
                    })
                except Exception as e:
                    print(f"FAILED GPR baseline {w} {test_cell}: {e}")
    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(RESULT_DIR, "approx_paper_gpr_ekf_leave_one_cell_results.csv"), index=False)
    if not res.empty:
        summary = res.groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]].agg(["mean", "std"]).reset_index()
    else:
        summary = pd.DataFrame()
    summary.to_csv(os.path.join(RESULT_DIR, "approx_paper_gpr_ekf_summary_mean_std.csv"), index=False)
    return res, summary

In [ ]:
# ============================================================
# 10) MULTI-HI TABULAR MODELS
# ============================================================
def evaluate_multi_hi_tabular(dataframe):
    rows = []
    feature_doc_rows = []
    models = make_tabular_models()
    cells = sorted(dataframe["cell"].unique())

    for exclude_outliers in EXCLUDE_OUTLIER_OPTIONS:
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()
        for w in WINDOWS:
            try:
                feature_cols = get_window_feature_cols(data_eval, w, USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
            except Exception as e:
                print(f"SKIPPED window {w}: {e}")
                continue
            for f in feature_cols:
                feature_doc_rows.append({"window": w, "feature": f, "exclude_outliers": exclude_outliers, "n_features": len(feature_cols)})
            for model_name, model in models.items():
                for test_cell in cells:
                    train_df = data_eval[data_eval["cell"] != test_cell].copy()
                    test_df = data_eval[data_eval["cell"] == test_cell].copy()
                    if len(train_df) < 20 or len(test_df) < 5:
                        continue
                    X_train = train_df[feature_cols]
                    y_train = train_df["soh"].values.astype(float)
                    X_test = test_df[feature_cols]
                    y_test = test_df["soh"].values.astype(float)
                    try:
                        m = clone(model)
                        m.fit(X_train, y_train)
                        pred = np.clip(m.predict(X_test), 0.5, 1.05)
                        rows.append({
                            "experiment": "Multi-HI-Tabular",
                            "window": w,
                            "test_cell": test_cell,
                            "model": model_name,
                            "exclude_outliers": exclude_outliers,
                            "n_train": len(train_df),
                            "n_test": len(test_df),
                            "n_features": len(feature_cols),
                            **regression_metrics(y_test, pred)
                        })
                    except Exception as e:
                        print(f"FAILED {w} {model_name} {test_cell}: {e}")
    res = pd.DataFrame(rows)
    feat_doc = pd.DataFrame(feature_doc_rows)
    res.to_csv(os.path.join(RESULT_DIR, "multi_hi_tabular_leave_one_cell_results.csv"), index=False)
    feat_doc.to_csv(os.path.join(RESULT_DIR, "leakage_clean_feature_sets.csv"), index=False)
    if not res.empty:
        summary = res.groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]].agg(["mean", "std"]).reset_index()
        best = res.groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]].mean().reset_index().sort_values(["exclude_outliers", "window", "RMSE"])
    else:
        summary = pd.DataFrame()
        best = pd.DataFrame()
    summary.to_csv(os.path.join(RESULT_DIR, "multi_hi_tabular_summary_mean_std.csv"), index=False)
    best.to_csv(os.path.join(RESULT_DIR, "best_models_ranked_by_window.csv"), index=False)
    return res, summary, best, feat_doc

In [ ]:
# ============================================================
# 11) MAW-GME STANDALONE
# ============================================================
def build_maw_gme_expert_specs(dataframe, scenario="all_windows_available", selected_names=None):
    base_models = make_tabular_models()
    if selected_names is None:
        selected_names = [n for n in MODEL_CANDIDATES if n in base_models]
    if scenario == "all_windows_available":
        expert_windows = list(WINDOWS.keys())
    elif scenario == "only_full_window":
        expert_windows = [first_window_by_prefix("full")]
    elif scenario == "only_medium_window":
        expert_windows = [first_window_by_prefix("medium")]
    elif scenario == "only_narrow_window":
        expert_windows = [first_window_by_prefix("narrow")]
    elif scenario == "only_very_narrow_window":
        expert_windows = [first_window_by_prefix("very_narrow")]
    elif scenario in ["pmax_features_missing", "ica_shape_features_missing"]:
        expert_windows = list(WINDOWS.keys())
    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    expert_specs = []
    for w in expert_windows:
        cols = get_window_feature_cols(dataframe, w, USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
        cols = apply_structured_feature_filter(cols, scenario)
        cols = remove_unusable_feature_cols(dataframe, cols, scenario_name=f"MAW-GME::{scenario}::{w}")
        for name in selected_names:
            expert_specs.append({"window": w, "model_name": name, "feature_cols": cols, "n_features": len(cols)})
    return expert_specs


def fit_predict_maw_gme(train_df, test_df, expert_specs, base_models):
    """Validation-weighted expert fusion. Test cell is not used for weight learning."""
    train_cells = sorted(train_df["cell"].unique())
    if len(train_cells) < 2:
        raise ValueError("MAW-GME needs at least two training cells.")
    val_cell = train_cells[-1]
    fit_df = train_df[train_df["cell"] != val_cell].copy()
    val_df = train_df[train_df["cell"] == val_cell].copy()

    y_fit = fit_df["soh"].values.astype(float)
    y_val = val_df["soh"].values.astype(float)
    y_train_full = train_df["soh"].values.astype(float)

    expert_preds = []
    expert_errors = []
    expert_info = []

    for spec in expert_specs:
        name = spec["model_name"]
        cols = spec["feature_cols"]
        try:
            val_model = clone(base_models[name])
            val_model.fit(fit_df[cols], y_fit)
            val_pred = np.clip(val_model.predict(val_df[cols]), 0.5, 1.05)
            val_rmse = float(np.sqrt(mean_squared_error(y_val, val_pred)))
            if not np.isfinite(val_rmse):
                continue
            final_model = clone(base_models[name])
            final_model.fit(train_df[cols], y_train_full)
            pred = np.clip(final_model.predict(test_df[cols]), 0.5, 1.05)
            expert_preds.append(pred)
            expert_errors.append(val_rmse)
            expert_info.append({"window": spec["window"], "model_name": name, "n_features": len(cols), "val_rmse": val_rmse})
        except Exception as e:
            print(f"FAILED expert {spec['window']} {name}: {e}")

    if len(expert_preds) == 0:
        raise ValueError("No valid expert predictions produced.")

    expert_preds = np.vstack(expert_preds)
    expert_errors = np.asarray(expert_errors)
    raw_weights = 1.0 / (expert_errors + 1e-8)
    weights = raw_weights / raw_weights.sum()
    final_pred = np.sum(expert_preds * weights[:, None], axis=0)
    for i, info in enumerate(expert_info):
        info["weight"] = float(weights[i])
    return np.clip(final_pred, 0.5, 1.05), expert_info


def evaluate_maw_gme(dataframe):
    rows = []
    cells = sorted(dataframe["cell"].unique())
    base_models = make_tabular_models()
    selected_names = [n for n in MODEL_CANDIDATES if n in base_models]

    for exclude_outliers in EXCLUDE_OUTLIER_OPTIONS:
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()
        expert_specs = build_maw_gme_expert_specs(data_eval, "all_windows_available", selected_names)
        for test_cell in cells:
            train_df = data_eval[data_eval["cell"] != test_cell].copy()
            test_df = data_eval[data_eval["cell"] == test_cell].copy()
            if len(train_df) < 20 or len(test_df) < 5:
                continue
            try:
                pred, _ = fit_predict_maw_gme(train_df, test_df, expert_specs, base_models)
                y_test = test_df["soh"].values.astype(float)
                rows.append({
                    "experiment": "MAW-GME",
                    "window": "adaptive_all_windows",
                    "test_cell": test_cell,
                    "model": "MAW-GME",
                    "exclude_outliers": exclude_outliers,
                    "n_train": len(train_df),
                    "n_test": len(test_df),
                    "n_features": len(get_all_window_feature_cols(data_eval)),
                    "n_experts": len(expert_specs),
                    **regression_metrics(y_test, pred)
                })
            except Exception as e:
                print(f"FAILED MAW-GME {test_cell}: {e}")
    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(RESULT_DIR, "maw_gme_leave_one_cell_results.csv"), index=False)
    if not res.empty:
        summary = res.groupby(["exclude_outliers", "window", "experiment", "model"])[["RMSE", "MAE", "R2"]].agg(["mean", "std"]).reset_index()
    else:
        summary = pd.DataFrame()
    summary.to_csv(os.path.join(RESULT_DIR, "maw_gme_summary_mean_std.csv"), index=False)
    return res, summary

In [ ]:
# ============================================================
# 12) STRUCTURED PARTIAL-WINDOW AND FEATURE-GROUP ABLATION + MAW-GME
# ============================================================
STRUCTURED_SCENARIOS = [
    "all_windows_available",
    "only_full_window",
    "only_medium_window",
    "only_narrow_window",
    "only_very_narrow_window",
    "pmax_features_missing",
    "ica_shape_features_missing",
]

SCENARIO_DESCRIPTIONS = {
    "all_windows_available": "All window-specific features are available.",
    "only_full_window": "Only full-window features are available.",
    "only_medium_window": "Only medium partial-window features are available.",
    "only_narrow_window": "Only narrow partial-window features are available.",
    "only_very_narrow_window": "Only very-narrow partial-window features are available.",
    "pmax_features_missing": "All pmax-related features are removed from the all-window feature set.",
    "ica_shape_features_missing": "ICA-shape/DVA features such as ic_std, peak_width, and DVA are removed.",
}


def get_structured_scenario_feature_cols(dataframe, scenario):
    if scenario == "all_windows_available":
        cols = get_all_window_feature_cols(dataframe)
    elif scenario == "only_full_window":
        cols = get_window_feature_cols(dataframe, first_window_by_prefix("full"), USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
    elif scenario == "only_medium_window":
        cols = get_window_feature_cols(dataframe, first_window_by_prefix("medium"), USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
    elif scenario == "only_narrow_window":
        cols = get_window_feature_cols(dataframe, first_window_by_prefix("narrow"), USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
    elif scenario == "only_very_narrow_window":
        cols = get_window_feature_cols(dataframe, first_window_by_prefix("very_narrow"), USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
    elif scenario in ["pmax_features_missing", "ica_shape_features_missing"]:
        cols = apply_structured_feature_filter(get_all_window_feature_cols(dataframe), scenario)
    else:
        raise ValueError(f"Unknown scenario: {scenario}")
    return remove_unusable_feature_cols(dataframe, cols, scenario_name=scenario)


def evaluate_structured_window_scenarios(dataframe):
    rows = []
    feature_rows = []
    weight_rows = []
    base_models = make_tabular_models()
    selected_names = [n for n in MODEL_CANDIDATES if n in base_models]
    cells = sorted(dataframe["cell"].unique())

    for exclude_outliers in EXCLUDE_OUTLIER_OPTIONS:
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

        for scenario in STRUCTURED_SCENARIOS:
            try:
                feature_cols = get_structured_scenario_feature_cols(data_eval, scenario)
            except Exception as e:
                print(f"SKIPPED scenario {scenario}: {e}")
                continue

            for f in feature_cols:
                feature_rows.append({
                    "scenario": scenario,
                    "description": SCENARIO_DESCRIPTIONS.get(scenario, ""),
                    "exclude_outliers": exclude_outliers,
                    "feature": f,
                    "n_features": len(feature_cols),
                })

            print(f"\nStructured scenario={scenario}, exclude_outliers={exclude_outliers}, n_features={len(feature_cols)}")

            # Standard tabular models
            for model_name in selected_names:
                for test_cell in cells:
                    train_df = data_eval[data_eval["cell"] != test_cell].copy()
                    test_df = data_eval[data_eval["cell"] == test_cell].copy()
                    if len(train_df) < 20 or len(test_df) < 5:
                        continue
                    try:
                        m = clone(base_models[model_name])
                        m.fit(train_df[feature_cols], train_df["soh"].values.astype(float))
                        pred = np.clip(m.predict(test_df[feature_cols]), 0.5, 1.05)
                        y_test = test_df["soh"].values.astype(float)
                        rows.append({
                            "experiment": "StructuredWindowAblation",
                            "scenario": scenario,
                            "description": SCENARIO_DESCRIPTIONS.get(scenario, ""),
                            "test_cell": test_cell,
                            "model": model_name,
                            "exclude_outliers": exclude_outliers,
                            "n_train": len(train_df),
                            "n_test": len(test_df),
                            "n_features": len(feature_cols),
                            "n_experts": np.nan,
                            **regression_metrics(y_test, pred)
                        })
                    except Exception as e:
                        print(f"FAILED structured {scenario} {model_name} {test_cell}: {e}")

            # Proposed MAW-GME under same scenario
            try:
                expert_specs = build_maw_gme_expert_specs(data_eval, scenario, selected_names)
                print(f"MAW-GME scenario={scenario}, exclude_outliers={exclude_outliers}, n_experts={len(expert_specs)}")
            except Exception as e:
                print(f"SKIPPED MAW-GME for scenario={scenario}: {e}")
                expert_specs = []

            for test_cell in cells:
                if len(expert_specs) == 0:
                    continue
                train_df = data_eval[data_eval["cell"] != test_cell].copy()
                test_df = data_eval[data_eval["cell"] == test_cell].copy()
                if len(train_df) < 20 or len(test_df) < 5:
                    continue
                try:
                    pred, expert_info = fit_predict_maw_gme(train_df, test_df, expert_specs, base_models)
                    y_test = test_df["soh"].values.astype(float)
                    rows.append({
                        "experiment": "StructuredWindowAblation",
                        "scenario": scenario,
                        "description": SCENARIO_DESCRIPTIONS.get(scenario, ""),
                        "test_cell": test_cell,
                        "model": "MAW-GME",
                        "exclude_outliers": exclude_outliers,
                        "n_train": len(train_df),
                        "n_test": len(test_df),
                        "n_features": len(feature_cols),
                        "n_experts": len(expert_specs),
                        **regression_metrics(y_test, pred)
                    })
                    for info in expert_info:
                        weight_rows.append({
                            "scenario": scenario,
                            "description": SCENARIO_DESCRIPTIONS.get(scenario, ""),
                            "exclude_outliers": exclude_outliers,
                            "test_cell": test_cell,
                            **info,
                        })
                except Exception as e:
                    print(f"FAILED structured MAW-GME {scenario} {test_cell}: {e}")

    res = pd.DataFrame(rows)
    feature_sets = pd.DataFrame(feature_rows)
    weights = pd.DataFrame(weight_rows)

    res.to_csv(os.path.join(RESULT_DIR, "structured_window_ablation_leave_one_cell_results.csv"), index=False)
    feature_sets.to_csv(os.path.join(RESULT_DIR, "structured_window_ablation_feature_sets.csv"), index=False)
    weights.to_csv(os.path.join(RESULT_DIR, "structured_window_ablation_maw_gme_expert_weights.csv"), index=False)

    if not res.empty:
        summary = res.groupby(["exclude_outliers", "scenario", "description", "model"])[["RMSE", "MAE", "R2"]].agg(["mean", "std"]).reset_index()
        best = res.groupby(["exclude_outliers", "scenario", "description", "model"])[["RMSE", "MAE", "R2"]].mean().reset_index().sort_values(["exclude_outliers", "scenario", "RMSE"])
    else:
        summary = pd.DataFrame()
        best = pd.DataFrame()
    summary.to_csv(os.path.join(RESULT_DIR, "structured_window_ablation_summary_mean_std.csv"), index=False)
    best.to_csv(os.path.join(RESULT_DIR, "structured_window_ablation_best_models.csv"), index=False)
    return res, summary, best, feature_sets, weights

In [ ]:
# ============================================================
# 13) DEEP SEQUENCE MODELS: LSTM, GRU, TCN, TRANSFORMER
# ============================================================
# This block is intentionally aligned with the Dataset 2 / Dataset 3 notebooks.
# It is optional because deep models are slower and usually weaker on small
# leave-one-cell-out battery datasets. For fair comparison, keep the same
# RUN_DEEP_MODELS setting across all dataset notebooks.

def evaluate_deep_models(dataframe, windows=WINDOWS, exclude_outliers=True):
    if not RUN_DEEP_MODELS:
        print("Deep sequence models skipped: RUN_DEEP_MODELS=False")
        return pd.DataFrame(), pd.DataFrame()

    try:
        import torch
        import torch.nn as nn
        from torch.utils.data import Dataset, DataLoader
    except Exception as e:
        print("Deep sequence models skipped because PyTorch is unavailable:", e)
        return pd.DataFrame(), pd.DataFrame()

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Deep learning device:", DEVICE)

    def set_torch_seed(seed=RANDOM_STATE):
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        np.random.seed(seed)
        random.seed(seed)

    class SeqDataset(Dataset):
        def __init__(self, X, y):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)
        def __len__(self):
            return len(self.y)
        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]

    def build_sequences(dataframe_in, feature_cols, seq_len=5):
        X_list, y_list, meta = [], [], []
        for cell, g in dataframe_in.sort_values(["cell", "cycle_number"]).groupby("cell"):
            g = g.dropna(subset=["soh"]).copy()
            if len(g) < seq_len:
                continue
            X = g[feature_cols].values.astype(float)
            y = g["soh"].values.astype(float)
            cycles = g["cycle_number"].values
            for i in range(seq_len - 1, len(g)):
                X_list.append(X[i-seq_len+1:i+1])
                y_list.append(y[i])
                meta.append({"cell": cell, "cycle_number": cycles[i]})
        return np.asarray(X_list, dtype=float), np.asarray(y_list, dtype=float), pd.DataFrame(meta)

    class LSTMRegressor(nn.Module):
        def __init__(self, n_features, hidden=32, layers=1, dropout=0.1):
            super().__init__()
            self.rnn = nn.LSTM(
                n_features, hidden, num_layers=layers, batch_first=True,
                dropout=dropout if layers > 1 else 0.0
            )
            self.head = nn.Sequential(
                nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1)
            )
        def forward(self, x):
            out, _ = self.rnn(x)
            return self.head(out[:, -1, :])

    class GRURegressor(nn.Module):
        def __init__(self, n_features, hidden=32, layers=1, dropout=0.1):
            super().__init__()
            self.rnn = nn.GRU(
                n_features, hidden, num_layers=layers, batch_first=True,
                dropout=dropout if layers > 1 else 0.0
            )
            self.head = nn.Sequential(
                nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1)
            )
        def forward(self, x):
            out, _ = self.rnn(x)
            return self.head(out[:, -1, :])

    class TCNRegressor(nn.Module):
        def __init__(self, n_features, hidden=32, dropout=0.1):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(n_features, hidden, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Conv1d(hidden, hidden, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
            self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
        def forward(self, x):
            z = x.permute(0, 2, 1)
            z = self.net(z)
            z = z[:, :, -1]
            return self.head(z)

    class TransformerRegressor(nn.Module):
        def __init__(self, n_features, d_model=32, nhead=4, num_layers=2, dropout=0.1):
            super().__init__()
            self.input_proj = nn.Linear(n_features, d_model)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=64,
                dropout=dropout,
                batch_first=True,
                activation="gelu",
            )
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.head = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1))
        def forward(self, x):
            z = self.input_proj(x)
            z = self.encoder(z)
            return self.head(z[:, -1, :])

    def train_torch_regressor(model, X_train, y_train, X_val, y_val, epochs=DEEP_EPOCHS):
        model = model.to(DEVICE)
        train_loader = DataLoader(SeqDataset(X_train, y_train), batch_size=DEEP_BATCH_SIZE, shuffle=True)
        val_X = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        val_y = torch.tensor(y_val, dtype=torch.float32).view(-1, 1).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=DEEP_LR, weight_decay=1e-4)
        loss_fn = nn.MSELoss()
        best_state = None
        best_val = float("inf")
        wait = 0

        for ep in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            model.eval()
            with torch.no_grad():
                val_loss = float(loss_fn(model(val_X), val_y).item())

            if val_loss < best_val - 1e-8:
                best_val = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= DEEP_PATIENCE:
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        return model

    def predict_torch(model, X):
        model.eval()
        with torch.no_grad():
            X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
            pred = model(X_t).detach().cpu().numpy().reshape(-1)
        return pred

    rows = []
    all_cells = sorted(dataframe["cell"].unique())
    data_eval = dataframe.copy()

    if exclude_outliers and "soh_outlier" in data_eval.columns:
        data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

    model_builders = {
        "LSTM": lambda nf: LSTMRegressor(nf),
        "GRU": lambda nf: GRURegressor(nf),
        "TCN": lambda nf: TCNRegressor(nf),
        "TransformerEncoder": lambda nf: TransformerRegressor(nf),
    }

    for window in windows:
        try:
            feature_cols = get_window_feature_cols(data_eval, window, USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
        except Exception as e:
            print(f"SKIPPED deep window={window}: {e}")
            continue

        for test_cell in all_cells:
            train_df = data_eval[data_eval["cell"] != test_cell].copy()
            test_df = data_eval[data_eval["cell"] == test_cell].copy()

            if len(train_df) < 20 or len(test_df) < DEEP_SEQ_LEN + 1:
                continue

            # Fit imputer/scaler only on train rows to avoid leakage.
            imputer = SimpleImputer(strategy="median")
            scaler = StandardScaler()
            imputer.fit(train_df[feature_cols])
            scaler.fit(imputer.transform(train_df[feature_cols]))

            train_df_proc = train_df.copy()
            test_df_proc = test_df.copy()
            train_scaled = scaler.transform(imputer.transform(train_df[feature_cols]))
            test_scaled = scaler.transform(imputer.transform(test_df[feature_cols]))
            train_df_proc.loc[:, feature_cols] = train_scaled
            test_df_proc.loc[:, feature_cols] = test_scaled

            train_cells = sorted(train_df_proc["cell"].unique())
            if len(train_cells) < 2:
                continue

            val_cell = train_cells[-1]
            proper_train = train_df_proc[train_df_proc["cell"] != val_cell].copy()
            val_df = train_df_proc[train_df_proc["cell"] == val_cell].copy()

            X_tr, y_tr, _ = build_sequences(proper_train, feature_cols, seq_len=DEEP_SEQ_LEN)
            X_val, y_val, _ = build_sequences(val_df, feature_cols, seq_len=DEEP_SEQ_LEN)
            X_te, y_te, _ = build_sequences(test_df_proc, feature_cols, seq_len=DEEP_SEQ_LEN)

            if len(X_tr) < 20 or len(X_val) < 5 or len(X_te) < 5:
                continue

            for model_name, builder in model_builders.items():
                try:
                    set_torch_seed(RANDOM_STATE)
                    model = builder(len(feature_cols))
                    model = train_torch_regressor(model, X_tr, y_tr, X_val, y_val, epochs=DEEP_EPOCHS)
                    pred = np.clip(predict_torch(model, X_te), 0.5, 1.05)
                    met = regression_metrics(y_te, pred)
                    rows.append({
                        "experiment": "MultiHI-DeepSequence",
                        "window": window,
                        "test_cell": test_cell,
                        "model": model_name,
                        "exclude_outliers": exclude_outliers,
                        "n_train": len(X_tr),
                        "n_test": len(X_te),
                        "n_features": len(feature_cols),
                        "seq_len": DEEP_SEQ_LEN,
                        **met,
                    })
                except Exception as e:
                    print(f"FAILED deep {model_name}, {window}, {test_cell}: {e}")

    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(RESULT_DIR, "multi_hi_deep_sequence_leave_one_cell_results.csv"), index=False)

    if not res.empty:
        summary = (
            res.groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
            .agg(["mean", "std"])
            .reset_index()
        )
    else:
        summary = pd.DataFrame()

    summary.to_csv(os.path.join(RESULT_DIR, "multi_hi_deep_sequence_summary_mean_std.csv"), index=False)
    return res, summary


In [ ]:
# ============================================================
# 13) SHAP ANALYSIS FOR REPRESENTATIVE TABULAR MODEL
# ============================================================
def run_shap_analysis(dataframe):
    if not RUN_SHAP or XGBRegressor is None:
        print("SHAP skipped: RUN_SHAP=False or XGBoost unavailable.")
        return pd.DataFrame()
    try:
        pip_install_if_missing("shap", "shap")
        import shap
    except Exception as e:
        print("SHAP import failed:", e)
        return pd.DataFrame()

    # Use all-window feature set and XGBoost for feature-level explanation.
    feature_cols = get_all_window_feature_cols(dataframe)
    data_eval = dataframe.copy()
    X = data_eval[feature_cols]
    y = data_eval["soh"].values.astype(float)

    imputer = SimpleImputer(strategy="median")
    X_imp = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols)
    model = XGBRegressor(
        n_estimators=350, learning_rate=0.03, max_depth=3,
        subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE,
        objective="reg:squarederror"
    )
    model.fit(X_imp, y)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_imp)
    mean_abs = np.abs(shap_values).mean(axis=0)
    imp = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap", ascending=False)
    imp.to_csv(os.path.join(RESULT_DIR, "shap_importance_advanced.csv"), index=False)

    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_imp, plot_type="bar", show=False, max_display=15)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "shap_bar_plot.png"), dpi=300, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_imp, show=False, max_display=15)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "shap_beeswarm_plot.png"), dpi=300, bbox_inches="tight")
    plt.close()

    if not imp.empty:
        top = imp.iloc[0]["feature"]
        plt.figure(figsize=(8, 6))
        shap.dependence_plot(top, shap_values, X_imp, show=False)
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "shap_dependence_top_feature.png"), dpi=300, bbox_inches="tight")
        plt.close()

    return imp

In [ ]:
# ============================================================
# 14) RUN EXPERIMENTS
# ============================================================
print("\n================ RUNNING DATASET 1 EXPERIMENTS ================")

single_hi_results, single_hi_summary = evaluate_single_hi_polynomial(df)
print("\nSingle-HI summary:")
display(single_hi_summary.head(30))

approx_gpr_results, approx_gpr_summary = evaluate_approx_paper_gpr(df)
print("\nApprox paper-like GPR summary:")
display(approx_gpr_summary.head(30))

multi_hi_results, multi_hi_summary, best_per_window, feature_doc = evaluate_multi_hi_tabular(df)
print("\nMulti-HI tabular summary:")
display(multi_hi_summary.head(30))

deep_results, deep_summary = evaluate_deep_models(df)
print("\nMulti-HI deep sequence summary:")
display(deep_summary.head(30) if isinstance(deep_summary, pd.DataFrame) and not deep_summary.empty else deep_summary)

maw_gme_results, maw_gme_summary = evaluate_maw_gme(df)
print("\nMAW-GME summary:")
display(maw_gme_summary)

structured_results, structured_summary, structured_best, structured_features, structured_weights = evaluate_structured_window_scenarios(df)
print("\nStructured scenario summary:")
display(structured_summary.head(50))
print("\nStructured best models:")
display(structured_best.groupby(["exclude_outliers", "scenario"]).head(8) if not structured_best.empty else structured_best)
print("\nStructured MAW-GME expert weights:")
display(structured_weights.head(50))

# Compare Single-HI vs best Multi-HI per matching window.
if not single_hi_results.empty and not multi_hi_results.empty:
    s = (
        single_hi_results
        .groupby(["exclude_outliers", "window"])["RMSE"]
        .mean()
        .reset_index(name="single_hi_rmse")
    )

    multi_candidates_for_comparison = [multi_hi_results]
    if "deep_results" in globals() and isinstance(deep_results, pd.DataFrame) and not deep_results.empty:
        multi_candidates_for_comparison.append(deep_results)

    multi_candidates_for_comparison = pd.concat(multi_candidates_for_comparison, ignore_index=True)

    m = (
        multi_candidates_for_comparison
        .groupby(["exclude_outliers", "window", "model"])["RMSE"]
        .mean()
        .reset_index()
    )

    idx = m.groupby(["exclude_outliers", "window"])["RMSE"].idxmin()
    mbest = m.loc[idx].rename(columns={"RMSE": "best_multi_hi_rmse", "model": "best_multi_hi_model"})
    single_vs_multi = s.merge(mbest, on=["exclude_outliers", "window"], how="left")
    single_vs_multi["rmse_improvement_percent"] = (
        100.0 *
        (single_vs_multi["single_hi_rmse"] - single_vs_multi["best_multi_hi_rmse"]) /
        single_vs_multi["single_hi_rmse"]
    )
else:
    single_vs_multi = pd.DataFrame()

single_vs_multi.to_csv(os.path.join(RESULT_DIR, "single_hi_vs_best_multi_hi_summary.csv"), index=False)
print("\nSingle-HI vs Best Multi-HI:")
display(single_vs_multi)

shap_importance = run_shap_analysis(df)
print("\nSHAP importance:")
display(shap_importance.head(20) if not shap_importance.empty else shap_importance)

# Combine main results.
all_model_results = pd.concat(
    [x for x in [single_hi_results, approx_gpr_results, multi_hi_results, deep_results, maw_gme_results, structured_results] if isinstance(x, pd.DataFrame) and not x.empty],
    ignore_index=True
)
all_model_results.to_csv(os.path.join(RESULT_DIR, "all_leave_one_cell_model_results.csv"), index=False)

if not all_model_results.empty:
    group_cols = [c for c in ["experiment", "window", "scenario", "model", "exclude_outliers"] if c in all_model_results.columns]
    all_summary = all_model_results.groupby(group_cols)[["RMSE", "MAE", "R2"]].agg(["mean", "std"]).reset_index()
else:
    all_summary = pd.DataFrame()
all_summary.to_csv(os.path.join(RESULT_DIR, "all_model_summary_mean_std.csv"), index=False)

In [ ]:
# ============================================================
# 15) EXPORT EXCEL REPORT
# ============================================================
def make_excel_safe(dfin):
    if dfin is None:
        return pd.DataFrame()
    dfx = dfin.copy()
    if isinstance(dfx.columns, pd.MultiIndex):
        dfx.columns = ["_".join([str(x) for x in col if str(x) != ""]).strip("_") for col in dfx.columns.to_flat_index()]
    else:
        dfx.columns = [str(c) for c in dfx.columns]
    if isinstance(dfx.index, pd.MultiIndex) or dfx.index.name is not None:
        dfx = dfx.reset_index()
    return dfx

excel_path = os.path.join(RESULT_DIR, "advanced_soh_experiment_report.xlsx")
if os.path.exists(excel_path):
    os.remove(excel_path)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for name, table in [
        ("cell_summary", cell_summary),
        ("all_cell_results", all_model_results),
        ("summary_mean_std", all_summary),
        ("single_hi_summary", single_hi_summary),
        ("approx_gpr_summary", approx_gpr_summary),
        ("multi_hi_summary", multi_hi_summary),
        ("deep_results", deep_results if "deep_results" in globals() else pd.DataFrame()),
        ("deep_summary", deep_summary if "deep_summary" in globals() else pd.DataFrame()),
        ("best_models_ranked", best_per_window),
        ("maw_gme_results", maw_gme_results),
        ("maw_gme_summary", maw_gme_summary),
        ("structured_results", structured_results),
        ("structured_summary", structured_summary),
        ("structured_best", structured_best),
        ("structured_features", structured_features),
        ("maw_gme_weights", structured_weights),
        ("single_vs_multi", single_vs_multi),
        ("feature_sets", feature_doc),
        ("correlation_with_soh", corr_df),
        ("shap_importance", shap_importance),
    ]:
        if isinstance(table, pd.DataFrame):
            make_excel_safe(table).to_excel(writer, sheet_name=name[:31], index=False)

print("\nDONE.")
print("All results saved to Google Drive folder:", RESULT_DIR)
print("Excel report:", excel_path)
print("Important outputs:")
for fn in [
    "multi_hi_features.csv",
    "single_hi_polynomial_ekf_leave_one_cell_results.csv",
    "single_hi_polynomial_ekf_summary_mean_std.csv",
    "approx_paper_gpr_ekf_leave_one_cell_results.csv",
    "approx_paper_gpr_ekf_summary_mean_std.csv",
    "multi_hi_tabular_leave_one_cell_results.csv",
    "multi_hi_tabular_summary_mean_std.csv",
    "multi_hi_deep_sequence_leave_one_cell_results.csv",
    "multi_hi_deep_sequence_summary_mean_std.csv",
    "maw_gme_leave_one_cell_results.csv",
    "maw_gme_summary_mean_std.csv",
    "structured_window_ablation_leave_one_cell_results.csv",
    "structured_window_ablation_summary_mean_std.csv",
    "structured_window_ablation_best_models.csv",
    "structured_window_ablation_feature_sets.csv",
    "structured_window_ablation_maw_gme_expert_weights.csv",
    "single_hi_vs_best_multi_hi_summary.csv",
    "shap_importance_advanced.csv",
    "all_leave_one_cell_model_results.csv",
    "all_model_summary_mean_std.csv",
    "advanced_soh_experiment_report.xlsx",
]:
    p = os.path.join(RESULT_DIR if fn != "multi_hi_features.csv" else FEATURE_DIR, fn)
    print(" -", p, "exists=", os.path.exists(p))

## BMS-oriented online evaluation added

This version preserves the original dataset paths and MAW-GME/Multi-HI structure, then adds two BMS-oriented protocols at the end of the notebook:

1. **BMS-1 online replay/runtime analysis**: chronological replay of the held-out cell with online feature availability checking and latency logging.
2. **BMS-2 confidence-aware online SoH tracker**: ageing-trend prior plus Multi-HI/MAW-GME measurement update.

Comparison is limited to the **best dataset-specific Multi-HI model** and the proposed **MAW-GME** model so the BMS tables stay clean.

Path note: Dataset 1 path is preserved from the old code: SEARCH_ROOT=/content/drive/MyDrive/Batarya/Dataset_1 with automatic detection of Cell01/03/10/11/12 folders.


In [ ]:

# ============================================================
# 14) BMS-ORIENTED ONLINE REPLAY + CONFIDENCE-AWARE TRACKER
# ============================================================
# This section adds two deployment-oriented BMS experiments:
#   BMS-1: chronological online replay/runtime analysis.
#   BMS-2: confidence-aware online SoH tracker.
#
# Comparison design:
#   1) Best dataset-specific Multi-HI model = accuracy-oriented deployment.
#   2) MAW-GME = proposed missing/window-aware ensemble deployment.
#
# Important: this is a BMS-like emulation/replay protocol, not a hardware BMS implementation.
# It uses the already extracted leakage-clean Multi-HI rows in chronological order.

import os
import time
import pickle
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATASET_LABEL = "Dataset 1"
BMS_RESULT_DIR = os.path.join(RESULT_DIR, "bms_online_results")
os.makedirs(BMS_RESULT_DIR, exist_ok=True)

# Fallback values are used only when automatic selection from multi_hi_results is unavailable.
BMS_FALLBACK_BEST_MODEL = "ExtraTrees"
BMS_FALLBACK_BEST_WINDOW = "full_32_42"

BMS_MIN_AVAILABILITY = 0.50       # minimum fraction of non-NaN features needed for a measurement update
BMS_AVAILABILITY_GAMMA = 2.0      # dynamic penalty for incomplete window features
BMS_EPS = 1e-8


def _safe_model_size_kb(obj):
    try:
        return float(len(pickle.dumps(obj)) / 1024.0)
    except Exception:
        return np.nan


def _bms_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    if ok.sum() == 0:
        return {"RMSE": np.nan, "MAE": np.nan, "R2": np.nan}
    y_true = y_true[ok]
    y_pred = y_pred[ok]
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)) if len(np.unique(y_true)) > 1 else np.nan,
    }


def _bms_availability(row, cols):
    if len(cols) == 0:
        return 0.0
    vals = row[cols]
    return float(vals.notna().mean())


def _resolve_best_multihi_model_window():
    """Pick the best offline Multi-HI model/window from the existing results, fallback otherwise."""
    model_name = BMS_FALLBACK_BEST_MODEL
    window_name = BMS_FALLBACK_BEST_WINDOW
    try:
        res = globals().get("multi_hi_results", None)
        if isinstance(res, pd.DataFrame) and not res.empty:
            tmp = res.copy()
            if "exclude_outliers" in tmp.columns:
                tmp = tmp[tmp["exclude_outliers"].astype(str).eq("False") | (tmp["exclude_outliers"] == False)].copy()
            if {"window", "model", "RMSE"}.issubset(tmp.columns) and not tmp.empty:
                ranked = (
                    tmp.groupby(["window", "model"], as_index=False)["RMSE"]
                    .mean()
                    .sort_values("RMSE")
                    .reset_index(drop=True)
                )
                available_models = make_tabular_models()
                ranked = ranked[ranked["model"].isin(available_models.keys())]
                if not ranked.empty:
                    model_name = str(ranked.iloc[0]["model"])
                    window_name = str(ranked.iloc[0]["window"])
                    print(f"BMS auto-selected best Multi-HI: {model_name} / {window_name} (mean RMSE={ranked.iloc[0]['RMSE']:.6f})")
                    return model_name, window_name
    except Exception as e:
        print("BMS best-model auto-selection failed; using fallback.", e)
    print(f"BMS fallback best Multi-HI: {model_name} / {window_name}")
    return model_name, window_name


def _fit_direct_deploy(train_df, model_name, window_name):
    """Fit one accuracy-oriented best Multi-HI model and estimate inner validation RMSE."""
    base_models = make_tabular_models()
    if model_name not in base_models:
        print(f"Requested model {model_name} not available. Falling back to ExtraTrees/Ridge.")
        model_name = "ExtraTrees" if "ExtraTrees" in base_models else list(base_models.keys())[0]

    cols = get_window_feature_cols(train_df, window_name, True, True)
    train_cells = sorted(train_df["cell"].unique())
    if len(train_cells) >= 2:
        val_cell = train_cells[-1]
        fit_df = train_df[train_df["cell"] != val_cell].copy()
        val_df = train_df[train_df["cell"] == val_cell].copy()
        val_avail = val_df.apply(lambda r: _bms_availability(r, cols), axis=1)
        val_df2 = val_df[val_avail >= BMS_MIN_AVAILABILITY].copy()
        if len(fit_df) >= 5 and len(val_df2) >= 2:
            try:
                vm = clone(base_models[model_name])
                vm.fit(fit_df[cols], fit_df["soh"].values.astype(float))
                vp = np.clip(vm.predict(val_df2[cols]), 0.5, 1.05)
                val_rmse = float(np.sqrt(mean_squared_error(val_df2["soh"].values.astype(float), vp)))
            except Exception:
                val_rmse = 0.02
        else:
            val_rmse = 0.02
    else:
        val_rmse = 0.02

    final_model = clone(base_models[model_name])
    final_model.fit(train_df[cols], train_df["soh"].values.astype(float))
    return {
        "kind": "Best-MultiHI",
        "model_name": model_name,
        "window": window_name,
        "model": final_model,
        "feature_cols": cols,
        "val_rmse": float(max(val_rmse, 1e-5)),
        "model_size_kb": _safe_model_size_kb(final_model),
    }


def _predict_direct_one(deploy, row):
    cols = deploy["feature_cols"]
    t0 = time.perf_counter()
    availability = _bms_availability(row, cols)
    X_one = pd.DataFrame([row[cols].to_dict()], columns=cols)
    feature_ms = (time.perf_counter() - t0) * 1000.0
    if availability < BMS_MIN_AVAILABILITY:
        return np.nan, availability, feature_ms, 0.0, "no_measurement"
    t1 = time.perf_counter()
    pred = float(np.clip(deploy["model"].predict(X_one)[0], 0.5, 1.05))
    infer_ms = (time.perf_counter() - t1) * 1000.0
    return pred, availability, feature_ms, infer_ms, "measurement_update"


def _fit_maw_gme_deploy(train_df):
    """Fit MAW-GME experts once for deployment and keep validation-derived weights."""
    base_models = make_tabular_models()
    selected_names = [n for n in globals().get("MODEL_CANDIDATES", ["Ridge", "ExtraTrees", "RandomForest", "HistGB", "XGBoost", "LightGBM", "CatBoost"]) if n in base_models]
    if len(selected_names) == 0:
        selected_names = list(base_models.keys())

    train_cells = sorted(train_df["cell"].unique())
    if len(train_cells) < 2:
        raise ValueError("MAW-GME deployment needs at least two training cells for inner validation.")
    val_cell = train_cells[-1]
    fit_df = train_df[train_df["cell"] != val_cell].copy()
    val_df = train_df[train_df["cell"] == val_cell].copy()

    experts = []

    for window_name in list(WINDOWS.keys()):
        try:
            cols = get_window_feature_cols(train_df, window_name, True, True)
        except Exception as e:
            print(f"MAW-GME BMS skipped window={window_name}: {e}")
            continue
        val_avail = val_df.apply(lambda r: _bms_availability(r, cols), axis=1)
        val_df2 = val_df[val_avail >= BMS_MIN_AVAILABILITY].copy()
        if len(val_df2) < 2 or len(fit_df) < 5:
            continue
        for model_name in selected_names:
            try:
                val_model = clone(base_models[model_name])
                val_model.fit(fit_df[cols], fit_df["soh"].values.astype(float))
                vp = np.clip(val_model.predict(val_df2[cols]), 0.5, 1.05)
                val_rmse = float(np.sqrt(mean_squared_error(val_df2["soh"].values.astype(float), vp)))
                if not np.isfinite(val_rmse):
                    continue
                final_model = clone(base_models[model_name])
                final_model.fit(train_df[cols], train_df["soh"].values.astype(float))
                base_weight = 1.0 / (val_rmse + BMS_EPS)
                experts.append({
                    "window": window_name,
                    "model_name": model_name,
                    "feature_cols": cols,
                    "model": final_model,
                    "val_rmse": float(max(val_rmse, 1e-5)),
                    "base_weight": float(base_weight),
                    "model_size_kb": _safe_model_size_kb(final_model),
                })
            except Exception as e:
                print(f"MAW-GME BMS expert failed window={window_name}, model={model_name}: {e}")

    if len(experts) == 0:
        raise ValueError("No valid MAW-GME BMS experts were fitted.")

    total_w = sum(e["base_weight"] for e in experts)
    for e in experts:
        e["base_weight_norm"] = float(e["base_weight"] / total_w)

    # Conservative validation RMSE proxy for the ensemble confidence.
    expert_rmses = np.array([e["val_rmse"] for e in experts], dtype=float)
    expert_weights = np.array([e["base_weight_norm"] for e in experts], dtype=float)
    ensemble_val_rmse = float(np.sum(expert_rmses * expert_weights))

    return {
        "kind": "MAW-GME",
        "model_name": "MAW-GME",
        "window": "adaptive_all_windows",
        "experts": experts,
        "val_rmse": float(max(ensemble_val_rmse, 1e-5)),
        "model_size_kb": float(np.nansum([e["model_size_kb"] for e in experts])),
        "n_experts": len(experts),
    }


def _predict_maw_gme_one(deploy, row):
    t0 = time.perf_counter()
    pred_items = []
    for e in deploy["experts"]:
        cols = e["feature_cols"]
        availability = _bms_availability(row, cols)
        if availability < BMS_MIN_AVAILABILITY:
            continue
        dynamic_weight = e["base_weight_norm"] * (availability ** BMS_AVAILABILITY_GAMMA)
        X_one = pd.DataFrame([row[cols].to_dict()], columns=cols)
        pred_items.append((e, X_one, availability, dynamic_weight))
    feature_ms = (time.perf_counter() - t0) * 1000.0
    if len(pred_items) == 0:
        return np.nan, 0.0, feature_ms, 0.0, "no_measurement"

    t1 = time.perf_counter()
    preds, weights, avails = [], [], []
    for e, X_one, availability, dynamic_weight in pred_items:
        try:
            p = float(np.clip(e["model"].predict(X_one)[0], 0.5, 1.05))
            preds.append(p)
            weights.append(dynamic_weight)
            avails.append(availability)
        except Exception:
            continue
    infer_ms = (time.perf_counter() - t1) * 1000.0
    if len(preds) == 0 or np.sum(weights) <= 0:
        return np.nan, 0.0, feature_ms, infer_ms, "no_measurement"
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    pred = float(np.sum(np.asarray(preds) * weights))
    return float(np.clip(pred, 0.5, 1.05)), float(np.mean(avails)), feature_ms, infer_ms, "measurement_update"


def _fit_prior_tracker(train_df):
    """Ageing-trend prior used when partial-window measurement is absent or uncertain."""
    train_cells = sorted(train_df["cell"].unique())
    prior_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0)),
    ])
    if len(train_cells) >= 2:
        val_cell = train_cells[-1]
        fit_df = train_df[train_df["cell"] != val_cell].copy()
        val_df = train_df[train_df["cell"] == val_cell].copy()
        try:
            prior_model.fit(fit_df[["cycle_number"]], fit_df["soh"].values.astype(float))
            vp = np.clip(prior_model.predict(val_df[["cycle_number"]]), 0.5, 1.05)
            val_rmse = float(np.sqrt(mean_squared_error(val_df["soh"].values.astype(float), vp)))
        except Exception:
            val_rmse = 0.03
    else:
        val_rmse = 0.03
    prior_model.fit(train_df[["cycle_number"]], train_df["soh"].values.astype(float))
    return {"model": prior_model, "val_rmse": float(max(val_rmse, 1e-5)), "model_size_kb": _safe_model_size_kb(prior_model)}


def _confidence_update(prior_pred, meas_pred, prior_rmse, meas_rmse, availability):
    if not np.isfinite(meas_pred) or availability < BMS_MIN_AVAILABILITY:
        return float(np.clip(prior_pred, 0.5, 1.05)), 0.0, "prediction_only"
    prior_var = float(prior_rmse ** 2 + BMS_EPS)
    meas_var = float((meas_rmse ** 2 + BMS_EPS) / (max(availability, BMS_EPS) ** 2))
    gain = prior_var / (prior_var + meas_var)
    gain = float(np.clip(gain, 0.0, 1.0))
    final = prior_pred + gain * (meas_pred - prior_pred)
    return float(np.clip(final, 0.5, 1.05)), gain, "measurement_update"


def evaluate_bms_online_protocols(dataframe):
    best_model_name, best_window_name = _resolve_best_multihi_model_window()
    cells = sorted(dataframe["cell"].unique())
    replay_rows = []
    tracker_rows = []

    data_eval = dataframe.copy()
    # Main BMS deployment uses the non-outlier-excluded setting by default.
    # This mirrors the main manuscript tables unless the user explicitly changes it.

    for test_cell in cells:
        train_df = data_eval[data_eval["cell"] != test_cell].copy()
        test_df = data_eval[data_eval["cell"] == test_cell].copy()
        if "cycle_number" in test_df.columns:
            test_df = test_df.sort_values("cycle_number").reset_index(drop=True)
        else:
            test_df = test_df.reset_index(drop=True)

        if len(train_df) < 20 or len(test_df) < 3:
            print(f"BMS skipped {test_cell}: insufficient train/test rows")
            continue

        print(f"\nBMS fold: held-out {test_cell}")
        direct_deploy = _fit_direct_deploy(train_df, best_model_name, best_window_name)
        maw_deploy = _fit_maw_gme_deploy(train_df)
        prior_deploy = _fit_prior_tracker(train_df)

        deploys = [direct_deploy, maw_deploy]

        for deploy in deploys:
            measurement_source = deploy["kind"] if deploy["kind"] == "MAW-GME" else f"Best-MultiHI::{deploy['model_name']}::{deploy['window']}"
            for idx, row in test_df.iterrows():
                true_soh = float(row["soh"])
                cycle_number = float(row["cycle_number"]) if "cycle_number" in row.index and pd.notna(row["cycle_number"]) else float(idx)

                if deploy["kind"] == "MAW-GME":
                    meas_pred, availability, feature_ms, infer_ms, replay_mode = _predict_maw_gme_one(deploy, row)
                else:
                    meas_pred, availability, feature_ms, infer_ms, replay_mode = _predict_direct_one(deploy, row)

                total_ms = feature_ms + infer_ms
                replay_rows.append({
                    "dataset": DATASET_LABEL,
                    "protocol": "BMS-1_online_replay",
                    "test_cell": test_cell,
                    "row_index": idx,
                    "cycle_number": cycle_number,
                    "measurement_source": measurement_source,
                    "model": deploy["model_name"],
                    "window": deploy["window"],
                    "mode": replay_mode,
                    "availability": availability,
                    "true_soh": true_soh,
                    "pred_soh": meas_pred,
                    "abs_error": abs(true_soh - meas_pred) if np.isfinite(meas_pred) else np.nan,
                    "feature_assembly_time_ms": feature_ms,
                    "inference_time_ms": infer_ms,
                    "total_update_time_ms": total_ms,
                    "model_size_kb": deploy["model_size_kb"],
                    "n_experts": deploy.get("n_experts", 1),
                    "measurement_val_rmse": deploy["val_rmse"],
                })

                prior_pred = float(np.clip(prior_deploy["model"].predict(pd.DataFrame([[cycle_number]], columns=["cycle_number"]))[0], 0.5, 1.05))
                tracker_pred, confidence_gain, tracker_mode = _confidence_update(
                    prior_pred=prior_pred,
                    meas_pred=meas_pred,
                    prior_rmse=prior_deploy["val_rmse"],
                    meas_rmse=deploy["val_rmse"],
                    availability=availability,
                )
                tracker_rows.append({
                    "dataset": DATASET_LABEL,
                    "protocol": "BMS-2_confidence_aware_tracker",
                    "test_cell": test_cell,
                    "row_index": idx,
                    "cycle_number": cycle_number,
                    "measurement_source": measurement_source,
                    "model": deploy["model_name"],
                    "window": deploy["window"],
                    "mode": tracker_mode,
                    "availability": availability,
                    "confidence_gain": confidence_gain,
                    "prior_soh": prior_pred,
                    "measurement_soh": meas_pred,
                    "true_soh": true_soh,
                    "pred_soh": tracker_pred,
                    "abs_error": abs(true_soh - tracker_pred),
                    "prior_val_rmse": prior_deploy["val_rmse"],
                    "measurement_val_rmse": deploy["val_rmse"],
                    "model_size_kb": deploy["model_size_kb"] + prior_deploy["model_size_kb"],
                    "n_experts": deploy.get("n_experts", 1),
                })

    replay = pd.DataFrame(replay_rows)
    tracker = pd.DataFrame(tracker_rows)

    def summarize_bms(dfin):
        if dfin.empty:
            return pd.DataFrame()
        summary_rows = []
        group_cols = ["dataset", "protocol", "measurement_source", "model", "window"]
        for keys, g in dfin.groupby(group_cols, dropna=False):
            y_true = g["true_soh"].values
            y_pred = g["pred_soh"].values
            metrics = _bms_regression_metrics(y_true, y_pred)
            mode_counts = g["mode"].value_counts().to_dict() if "mode" in g.columns else {}
            row = dict(zip(group_cols, keys))
            row.update(metrics)
            row.update({
                "n_updates": int(np.isfinite(g["pred_soh"]).sum()),
                "n_rows": int(len(g)),
                "measurement_update_count": int(mode_counts.get("measurement_update", 0)),
                "prediction_only_count": int(mode_counts.get("prediction_only", 0)),
                "no_measurement_count": int(mode_counts.get("no_measurement", 0)),
                "mean_availability": float(g["availability"].mean()),
                "mean_model_size_kb": float(g["model_size_kb"].mean()),
                "mean_n_experts": float(g["n_experts"].mean()),
            })
            for c in ["feature_assembly_time_ms", "inference_time_ms", "total_update_time_ms", "confidence_gain"]:
                if c in g.columns:
                    row[f"mean_{c}"] = float(g[c].mean())
                    row[f"median_{c}"] = float(g[c].median())
            summary_rows.append(row)
        return pd.DataFrame(summary_rows).sort_values(["protocol", "measurement_source"]).reset_index(drop=True)

    replay_summary = summarize_bms(replay)
    tracker_summary = summarize_bms(tracker)
    combined_summary = pd.concat([replay_summary, tracker_summary], ignore_index=True)

    replay.to_csv(os.path.join(BMS_RESULT_DIR, "bms1_online_replay_results.csv"), index=False)
    replay_summary.to_csv(os.path.join(BMS_RESULT_DIR, "bms1_online_replay_summary.csv"), index=False)
    tracker.to_csv(os.path.join(BMS_RESULT_DIR, "bms2_confidence_tracker_results.csv"), index=False)
    tracker_summary.to_csv(os.path.join(BMS_RESULT_DIR, "bms2_confidence_tracker_summary.csv"), index=False)
    combined_summary.to_csv(os.path.join(BMS_RESULT_DIR, "bms_combined_summary.csv"), index=False)

    try:
        with pd.ExcelWriter(os.path.join(BMS_RESULT_DIR, "bms_online_results.xlsx")) as writer:
            replay.to_excel(writer, sheet_name="bms1_replay_rows", index=False)
            replay_summary.to_excel(writer, sheet_name="bms1_summary", index=False)
            tracker.to_excel(writer, sheet_name="bms2_tracker_rows", index=False)
            tracker_summary.to_excel(writer, sheet_name="bms2_summary", index=False)
            combined_summary.to_excel(writer, sheet_name="combined_summary", index=False)
    except Exception as e:
        print("Excel export skipped:", e)

    print("\nBMS-1 online replay summary:")
    display(replay_summary)
    print("\nBMS-2 confidence-aware tracker summary:")
    display(tracker_summary)
    print("\nBMS results saved to:", BMS_RESULT_DIR)
    return replay, replay_summary, tracker, tracker_summary, combined_summary


bms1_replay_results, bms1_replay_summary, bms2_tracker_results, bms2_tracker_summary, bms_combined_summary = evaluate_bms_online_protocols(df)


## BMS-3 missing-window / intermittent HI stress test

This additional BMS-oriented block adds two stress tests for the final deployment comparison:

1. **BMS-3A Kadem-style intermittent HI availability:** measurement updates are available with probabilities 100%, 80%, 60%, and 40%. When the measurement is unavailable, the tracker falls back to the ageing-trend prior.
2. **BMS-3B structured missing-window stress:** voltage-window availability is restricted deterministically, so MAW-GME can be evaluated under realistic partial-window conditions rather than random feature deletion.

The comparison is limited to the selected best Multi-HI model and the proposed MAW-GME deployment model.


In [ ]:

# ============================================================
# 15) BMS-3: MISSING-WINDOW / INTERMITTENT HI STRESS TEST
# ============================================================
# This section extends the BMS-oriented evaluation with two stress tests:
#
#   BMS-3A: Kadem-style intermittent HI availability
#       - Measurement-update availability probabilities: 100%, 80%, 60%, 40%.
#       - When an HI/window measurement is unavailable, the tracker uses prediction-only mode.
#       - This supports a fairer environment-level comparison with the single-HI GPR-EKF idea.
#
#   BMS-3B: Structured missing-window stress
#       - Window availability is restricted deterministically.
#       - This is closer to partial-charging behaviour than random feature deletion.
#       - Best Multi-HI can only update when its selected window is available.
#       - MAW-GME can switch to the available window/model experts.
#
# Notes:
#   - This is still a BMS-like replay/stress protocol, not a hardware BMS implementation.
#   - It reuses the model-selection and deployment helpers defined in the BMS-1/BMS-2 cell.
#   - Final models are trained only on training cells in each leave-one-cell-out fold.

import os
import time
import numpy as np
import pandas as pd
from IPython.display import display

BMS3_AVAILABILITY_PROBS = [1.0, 0.8, 0.6, 0.4]
BMS3_RANDOM_SEEDS = [11, 29, 47, 71, 101]
BMS3_RUN_STRUCTURED_WINDOW_STRESS = True
BMS3_RESULT_PREFIX = "bms3"


def _bms3_predict_maw_gme_one_with_allowed_windows(deploy, row, allowed_windows=None):
    """MAW-GME prediction with optional deterministic window masking."""
    allowed_windows = None if allowed_windows is None else set(allowed_windows)
    t0 = time.perf_counter()
    pred_items = []
    for e in deploy["experts"]:
        if allowed_windows is not None and e["window"] not in allowed_windows:
            continue
        cols = e["feature_cols"]
        availability = _bms_availability(row, cols)
        if availability < BMS_MIN_AVAILABILITY:
            continue
        dynamic_weight = e["base_weight_norm"] * (availability ** BMS_AVAILABILITY_GAMMA)
        X_one = pd.DataFrame([row[cols].to_dict()], columns=cols)
        pred_items.append((e, X_one, availability, dynamic_weight))
    feature_ms = (time.perf_counter() - t0) * 1000.0

    if len(pred_items) == 0:
        return np.nan, 0.0, feature_ms, 0.0, "no_measurement"

    t1 = time.perf_counter()
    preds, weights, avails = [], [], []
    for e, X_one, availability, dynamic_weight in pred_items:
        try:
            p = float(np.clip(e["model"].predict(X_one)[0], 0.5, 1.05))
            preds.append(p)
            weights.append(dynamic_weight)
            avails.append(availability)
        except Exception:
            continue
    infer_ms = (time.perf_counter() - t1) * 1000.0

    if len(preds) == 0 or np.sum(weights) <= 0:
        return np.nan, 0.0, feature_ms, infer_ms, "no_measurement"

    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    pred = float(np.sum(np.asarray(preds, dtype=float) * weights))
    return float(np.clip(pred, 0.5, 1.05)), float(np.mean(avails)), feature_ms, infer_ms, "measurement_update"


def _bms3_predict_deploy_one(deploy, row, allowed_windows=None):
    """Prediction wrapper for Best Multi-HI or MAW-GME with optional window masking."""
    allowed_windows = None if allowed_windows is None else set(allowed_windows)
    if deploy["kind"] == "MAW-GME":
        return _bms3_predict_maw_gme_one_with_allowed_windows(deploy, row, allowed_windows=allowed_windows)

    # Best Multi-HI uses one selected window. If that window is not physically available,
    # the model cannot perform a measurement update in this stress scenario.
    if allowed_windows is not None and deploy["window"] not in allowed_windows:
        return np.nan, 0.0, 0.0, 0.0, "no_measurement"
    return _predict_direct_one(deploy, row)


def _bms3_measurement_source_name(deploy):
    return deploy["kind"] if deploy["kind"] == "MAW-GME" else f"Best-MultiHI::{deploy['model_name']}::{deploy['window']}"


def _bms3_build_structured_window_scenarios(best_window_name):
    window_names = list(WINDOWS.keys())
    scenarios = []
    scenarios.append(("all_windows_available", window_names))
    scenarios.append(("best_window_only", [best_window_name]))
    for w in window_names:
        scenarios.append((f"only_{w}", [w]))
    if len(window_names) > 1:
        scenarios.append(("best_window_unavailable", [w for w in window_names if w != best_window_name]))

    # Remove duplicate scenario names while preserving order.
    out, seen = [], set()
    for name, wins in scenarios:
        if name not in seen:
            out.append((name, wins))
            seen.add(name)
    return out


def _bms3_summarize(dfin, extra_group_cols=None):
    if dfin is None or dfin.empty:
        return pd.DataFrame()
    extra_group_cols = extra_group_cols or []
    group_cols = ["dataset", "protocol", "measurement_source", "model", "window"] + extra_group_cols
    rows = []
    for keys, g in dfin.groupby(group_cols, dropna=False):
        metrics = _bms_regression_metrics(g["true_soh"].values, g["pred_soh"].values)
        mode_counts = g["mode"].value_counts().to_dict() if "mode" in g.columns else {}
        row = dict(zip(group_cols, keys))
        row.update(metrics)
        row.update({
            "n_rows": int(len(g)),
            "n_predictions": int(np.isfinite(g["pred_soh"]).sum()),
            "measurement_update_count": int(mode_counts.get("measurement_update", 0)),
            "prediction_only_count": int(mode_counts.get("prediction_only", 0)),
            "no_measurement_count": int(mode_counts.get("no_measurement", 0)),
            "mean_physical_availability": float(g["physical_availability"].mean()) if "physical_availability" in g.columns else np.nan,
            "mean_effective_availability": float(g["effective_availability"].mean()) if "effective_availability" in g.columns else np.nan,
            "mean_confidence_gain": float(g["confidence_gain"].mean()) if "confidence_gain" in g.columns else np.nan,
            "mean_model_size_kb": float(g["model_size_kb"].mean()) if "model_size_kb" in g.columns else np.nan,
            "mean_n_experts": float(g["n_experts"].mean()) if "n_experts" in g.columns else np.nan,
        })
        if "measurement_dropped" in g.columns:
            row["measurement_dropped_count"] = int(g["measurement_dropped"].sum())
            row["measurement_dropped_rate"] = float(g["measurement_dropped"].mean())
        if "seed" in g.columns:
            row["n_seeds"] = int(g["seed"].nunique())
        rows.append(row)
    return pd.DataFrame(rows).sort_values(group_cols).reset_index(drop=True)


def evaluate_bms3_missing_window_stress(dataframe):
    best_model_name, best_window_name = _resolve_best_multihi_model_window()
    cells = sorted(dataframe["cell"].unique())

    intermittent_rows = []
    structured_rows = []

    for fold_idx, test_cell in enumerate(cells):
        train_df = dataframe[dataframe["cell"] != test_cell].copy()
        test_df = dataframe[dataframe["cell"] == test_cell].copy()
        if "cycle_number" in test_df.columns:
            test_df = test_df.sort_values("cycle_number").reset_index(drop=True)
        else:
            test_df = test_df.reset_index(drop=True)

        if len(train_df) < 20 or len(test_df) < 3:
            print(f"BMS-3 skipped {test_cell}: insufficient train/test rows")
            continue

        print(f"\nBMS-3 fold: held-out {test_cell}")
        direct_deploy = _fit_direct_deploy(train_df, best_model_name, best_window_name)
        maw_deploy = _fit_maw_gme_deploy(train_df)
        prior_deploy = _fit_prior_tracker(train_df)
        deploys = [direct_deploy, maw_deploy]

        # Precompute normal measurements once per fold/model to avoid repeated expert inference for each random seed.
        cached_measurements = {}
        for deploy_idx, deploy in enumerate(deploys):
            source_name = _bms3_measurement_source_name(deploy)
            cache = []
            for idx, row in test_df.iterrows():
                true_soh = float(row["soh"])
                cycle_number = float(row["cycle_number"]) if "cycle_number" in row.index and pd.notna(row["cycle_number"]) else float(idx)
                prior_pred = float(np.clip(prior_deploy["model"].predict(pd.DataFrame([[cycle_number]], columns=["cycle_number"]))[0], 0.5, 1.05))
                meas_pred, physical_availability, feature_ms, infer_ms, meas_mode = _bms3_predict_deploy_one(deploy, row, allowed_windows=None)
                cache.append({
                    "row_index": idx,
                    "cycle_number": cycle_number,
                    "true_soh": true_soh,
                    "prior_soh": prior_pred,
                    "measurement_soh": meas_pred,
                    "physical_availability": physical_availability,
                    "feature_assembly_time_ms": feature_ms,
                    "inference_time_ms": infer_ms,
                    "measurement_mode": meas_mode,
                })
            cached_measurements[source_name] = (deploy, cache)

        # ------------------------------
        # BMS-3A: intermittent HI/update availability stress
        # ------------------------------
        for deploy_idx, deploy in enumerate(deploys):
            source_name = _bms3_measurement_source_name(deploy)
            _, cache = cached_measurements[source_name]
            for availability_prob in BMS3_AVAILABILITY_PROBS:
                for seed in BMS3_RANDOM_SEEDS:
                    rng = np.random.default_rng(int(seed + 1000 * availability_prob + 100 * fold_idx + 10 * deploy_idx))
                    for item in cache:
                        physical_ok = np.isfinite(item["measurement_soh"]) and item["measurement_mode"] == "measurement_update" and item["physical_availability"] >= BMS_MIN_AVAILABILITY
                        keep_measurement = physical_ok and ((availability_prob >= 0.999999) or (rng.random() < availability_prob))
                        effective_availability = float(item["physical_availability"] if keep_measurement else 0.0)
                        measurement_for_update = float(item["measurement_soh"]) if keep_measurement else np.nan
                        tracker_pred, confidence_gain, mode = _confidence_update(
                            prior_pred=item["prior_soh"],
                            meas_pred=measurement_for_update,
                            prior_rmse=prior_deploy["val_rmse"],
                            meas_rmse=deploy["val_rmse"],
                            availability=effective_availability,
                        )
                        intermittent_rows.append({
                            "dataset": DATASET_LABEL,
                            "protocol": "BMS-3A_intermittent_HI_stress",
                            "test_cell": test_cell,
                            "row_index": item["row_index"],
                            "cycle_number": item["cycle_number"],
                            "measurement_source": source_name,
                            "model": deploy["model_name"],
                            "window": deploy["window"],
                            "availability_probability": float(availability_prob),
                            "seed": int(seed),
                            "mode": mode,
                            "measurement_dropped": bool(physical_ok and not keep_measurement),
                            "physical_availability": float(item["physical_availability"]),
                            "effective_availability": effective_availability,
                            "confidence_gain": float(confidence_gain),
                            "prior_soh": float(item["prior_soh"]),
                            "measurement_soh": item["measurement_soh"],
                            "true_soh": float(item["true_soh"]),
                            "pred_soh": float(tracker_pred),
                            "abs_error": abs(float(item["true_soh"]) - float(tracker_pred)),
                            "feature_assembly_time_ms": item["feature_assembly_time_ms"],
                            "inference_time_ms": item["inference_time_ms"],
                            "total_update_time_ms": item["feature_assembly_time_ms"] + item["inference_time_ms"],
                            "prior_val_rmse": prior_deploy["val_rmse"],
                            "measurement_val_rmse": deploy["val_rmse"],
                            "model_size_kb": deploy["model_size_kb"] + prior_deploy["model_size_kb"],
                            "n_experts": deploy.get("n_experts", 1),
                        })

        # ------------------------------
        # BMS-3B: structured missing-window stress
        # ------------------------------
        if BMS3_RUN_STRUCTURED_WINDOW_STRESS:
            window_scenarios = _bms3_build_structured_window_scenarios(best_window_name)
            for scenario_name, allowed_windows in window_scenarios:
                for deploy in deploys:
                    source_name = _bms3_measurement_source_name(deploy)
                    for idx, row in test_df.iterrows():
                        true_soh = float(row["soh"])
                        cycle_number = float(row["cycle_number"]) if "cycle_number" in row.index and pd.notna(row["cycle_number"]) else float(idx)
                        prior_pred = float(np.clip(prior_deploy["model"].predict(pd.DataFrame([[cycle_number]], columns=["cycle_number"]))[0], 0.5, 1.05))
                        meas_pred, physical_availability, feature_ms, infer_ms, meas_mode = _bms3_predict_deploy_one(deploy, row, allowed_windows=allowed_windows)
                        effective_availability = float(physical_availability if np.isfinite(meas_pred) and meas_mode == "measurement_update" else 0.0)
                        tracker_pred, confidence_gain, mode = _confidence_update(
                            prior_pred=prior_pred,
                            meas_pred=meas_pred,
                            prior_rmse=prior_deploy["val_rmse"],
                            meas_rmse=deploy["val_rmse"],
                            availability=effective_availability,
                        )
                        structured_rows.append({
                            "dataset": DATASET_LABEL,
                            "protocol": "BMS-3B_structured_missing_window_stress",
                            "test_cell": test_cell,
                            "row_index": idx,
                            "cycle_number": cycle_number,
                            "measurement_source": source_name,
                            "model": deploy["model_name"],
                            "window": deploy["window"],
                            "stress_scenario": scenario_name,
                            "allowed_windows": ";".join(allowed_windows),
                            "mode": mode,
                            "physical_availability": float(physical_availability),
                            "effective_availability": effective_availability,
                            "confidence_gain": float(confidence_gain),
                            "prior_soh": prior_pred,
                            "measurement_soh": meas_pred,
                            "true_soh": true_soh,
                            "pred_soh": float(tracker_pred),
                            "abs_error": abs(true_soh - float(tracker_pred)),
                            "feature_assembly_time_ms": feature_ms,
                            "inference_time_ms": infer_ms,
                            "total_update_time_ms": feature_ms + infer_ms,
                            "prior_val_rmse": prior_deploy["val_rmse"],
                            "measurement_val_rmse": deploy["val_rmse"],
                            "model_size_kb": deploy["model_size_kb"] + prior_deploy["model_size_kb"],
                            "n_experts": deploy.get("n_experts", 1),
                        })

    intermittent = pd.DataFrame(intermittent_rows)
    structured = pd.DataFrame(structured_rows)

    intermittent_summary = _bms3_summarize(intermittent, extra_group_cols=["availability_probability"])
    structured_summary = _bms3_summarize(structured, extra_group_cols=["stress_scenario"]) if not structured.empty else pd.DataFrame()

    # Save outputs.
    intermittent.to_csv(os.path.join(BMS_RESULT_DIR, "bms3_intermit_hi_results.csv"), index=False)
    intermittent_summary.to_csv(os.path.join(BMS_RESULT_DIR, "bms3_intermit_hi_summary.csv"), index=False)
    structured.to_csv(os.path.join(BMS_RESULT_DIR, "bms3_structured_missing_window_results.csv"), index=False)
    structured_summary.to_csv(os.path.join(BMS_RESULT_DIR, "bms3_structured_missing_window_summary.csv"), index=False)

    # Combine summaries with BMS-1/BMS-2 if they were already computed in the previous cell.
    summary_frames = []
    for name in ["bms1_replay_summary", "bms2_tracker_summary"]:
        obj = globals().get(name, None)
        if isinstance(obj, pd.DataFrame) and not obj.empty:
            summary_frames.append(obj.copy())
    if not intermittent_summary.empty:
        summary_frames.append(intermittent_summary.copy())
    if not structured_summary.empty:
        summary_frames.append(structured_summary.copy())
    all_bms_summary = pd.concat(summary_frames, ignore_index=True, sort=False) if summary_frames else pd.DataFrame()
    all_bms_summary.to_csv(os.path.join(BMS_RESULT_DIR, "bms_all_protocols_summary.csv"), index=False)

    try:
        with pd.ExcelWriter(os.path.join(BMS_RESULT_DIR, "bms_online_results_with_bms3.xlsx")) as writer:
            # Keep row sheets separate to avoid huge, unwieldy workbooks.
            if isinstance(globals().get("bms1_replay_summary", None), pd.DataFrame):
                globals()["bms1_replay_summary"].to_excel(writer, sheet_name="bms1_summary", index=False)
            if isinstance(globals().get("bms2_tracker_summary", None), pd.DataFrame):
                globals()["bms2_tracker_summary"].to_excel(writer, sheet_name="bms2_summary", index=False)
            intermittent_summary.to_excel(writer, sheet_name="bms3_intermit_sum", index=False)
            structured_summary.to_excel(writer, sheet_name="bms3_window_sum", index=False)
            all_bms_summary.to_excel(writer, sheet_name="all_protocols_summary", index=False)
            # Row-level outputs can be large, but are useful for auditing.
            intermittent.to_excel(writer, sheet_name="bms3_intermit_rows", index=False)
            if not structured.empty:
                structured.to_excel(writer, sheet_name="bms3_window_rows", index=False)
    except Exception as e:
        print("BMS-3 Excel export skipped:", e)

    print("\nBMS-3A intermittent HI availability summary:")
    display(intermittent_summary)
    if not structured_summary.empty:
        print("\nBMS-3B structured missing-window summary:")
        display(structured_summary)
    print("\nBMS-3 results saved to:", BMS_RESULT_DIR)

    return intermittent, intermittent_summary, structured, structured_summary, all_bms_summary


bms3_intermit_hi_results, bms3_intermit_hi_summary, bms3_structured_missing_window_results, bms3_structured_missing_window_summary, bms_all_protocols_summary = evaluate_bms3_missing_window_stress(df)
